# ARC-AGI-3 · klee A — solo solution, public LB 39.22

**Scored version: version 25 of this notebook** (submission 57745091, public 39.22). This latest version is the
same code and settings in readable form: the base64 blobs of v25 are now plain files in the dataset
[`zhangzibo54/arc3-kleea-solution-code`](https://www.kaggle.com/datasets/zhangzibo54/arc3-kleea-solution-code).
Offline it plays 10 public games for 10 minutes each.

**Code, experiment log and lessons: [https://github.com/zhangzibo54-cmd/arc3-kleea-solution](https://github.com/zhangzibo54-cmd/arc3-kleea-solution)**
(`experiments/SUBMISSIONS.md`, `experiments/LESSONS.md`, `docs/SWITCHES.md`).

### What we changed
Agent: [The Duck](https://github.com/Tufalabs/duck-harness) (Tufa Labs) with
[Daniel Franzen's Milestone-2 solution](https://github.com/da-fr/arc-agi-3-solution) (public 28.17 as our baseline).

**1. Throughput (most of the gain).** One RTX PRO 6000, SGLang (Pennyroyal v2.5.3):
- *Hybrid A* weights: 3-bit routed experts (klee100 AutoRound 3bpw) in layers 0–11 and 36–47, Intel W4A16 in
  layers 12–35, Humming W3/W4 kernels → model ~70 → ~55 GB.
- KV in NVFP4 on the QSA layers 15–35, FP8 on the first/last QSA layers (NVFP4 everywhere lost points).
- 20 concurrent games at 148K context (Franzen: 10); ~1,000 tok/s aggregate decode vs ~600.
- Small decode kernels (split-K BF16 GEMM, fused HC mix/combine-norm), M-RoPE device cache, image cache.

**2. Harness** (34 switches on top of Franzen's, all listed in `docs/SWITCHES.md`):
stuck-level note after 30K tokens on a level with a checklist of causes; a lab-notebook summary kept in the
system prompt across context trims; object-level frame diffs and long-animation notes; history deltas instead of
full runtime state; no-op action limit; 30 s Python limit.

Online: 28.17 → 32.56 (history deltas) → 34.66 (hybrid A, 18 streams) → 37.96 (stuck-level note) →
**39.22** (mixed NVFP4/FP8 KV, 20 streams).


## 0. Harness patch

Our changes to the Tufa harness, from the code dataset.

In [ ]:
# Plain-file sources (formerly base64 blobs): the dataset zhangzibo54/arc3-kleea-solution-code holds
# serving/ and patches/ of the GitHub repo, flattened.
from pathlib import Path as _P

_CODE_DIRS = [_P('/kaggle/input/datasets/zhangzibo54/arc3-kleea-solution-code'),
              _P('/kaggle/input/arc3-kleea-solution-code')]


def code_file(name: str) -> bytes:
    for d in _CODE_DIRS:
        hits = sorted(d.rglob(name)) if d.is_dir() else []
        if hits:
            return hits[0].read_bytes()
    raise FileNotFoundError(f'{name} not found in the arc3-kleea-solution-code dataset')


# The harness diff (Duck base -> our harness): patches/harness-changes.v25.patch
_P('/kaggle/harness-changes.patch').write_bytes(code_file('harness-changes.v25.patch'))
assert __import__('hashlib').sha256(open('/kaggle/harness-changes.patch', 'rb').read()).hexdigest() == '174a9163b51eab6ab28ae3ebe7684baa964761aa7f22a1235ecd7a34b904c963'


## 1. Environment and submission mode

Detect whether this is a real competition rerun (which minimises diagnostics), set the
framework's environment flags, and put the CUDA libraries on the linker path.

In [ ]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# Kaggle might cancel the run in no action is done in some timeframe.
# To prevent this, harness will start up even if backend is not yet ready,
# and the first started game will perform a single reset action.
NOTEBOOK_START_TIME = time.time()
SERVER_STARTUP_TIMEOUT = 12 * 60
os.environ['ARC3_WARMUP_ACTION_GAMES'] = '1'
os.environ['ARC3_HTTP_RETRY_INITIAL_SECONDS'] = '900'

# model and dataset paths
WHEELHOUSE_DIR    = '/kaggle/input/datasets/dfranzen/pennyroyal-v253'
MODEL_DIR         = '/kaggle/input/models/dfranzen/intel-qwen3.8-flash-next-w4a16-autoround/transformers/default/1'
DRAFT_MODEL_DIR   = '/kaggle/input/models/dfranzen/albucino-qwen3-8-flash-next-drafter/transformers/default/1'
ORIG_BUNDLE_DIR   = '/kaggle/input/datasets/dfranzen/taaf-kaggle-source-bundle-copy'
SERVED_MODEL_NAME = 'flashnext'

# Apply harness patch
BUNDLE_DIR = Path('/kaggle/taaf-kaggle-source-share')
!rm -Rf $BUNDLE_DIR
!cp -a $ORIG_BUNDLE_DIR $BUNDLE_DIR
subprocess.run(["git", "apply", "--include=ARC3-Inference/*", "-v", "/kaggle/harness-changes.patch"], cwd=f"{BUNDLE_DIR}/src", check=True)
print('harness patch applied successfully')

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# In submission, disable the periodic JSON/HTML diagnostics writes and per-frame logging.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Everything the run produces is written here.
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")
SERVED_MODEL_HOST = '127.0.0.1'
SERVED_MODEL_PORT = 8001
SERVER_BASE_URL = f'http://{SERVED_MODEL_HOST}:{SERVED_MODEL_PORT}/v1'
USE_PRIORITY_SCHEDULING = True

setup_env = {
    'ARC3_LEVEL_FEEDBACK': '1',
    'ARC3_SEARCH_PUSH2': '1',  # baseline search push2
    ########## transformers config ##########
    'USE_TF': '0',
    'TRANSFORMERS_NO_TF': '1',
    'TRANSFORMERS_NO_TORCHVISION': '1',

    ########## serving config ##########
    'LOCAL_ANALYZER_BASE_URL': SERVER_BASE_URL,
    'OPENAI_BASE_URL': SERVER_BASE_URL,
    'LOCAL_ANALYZER_PROVIDER': 'vllm',
    'OPENAI_PROVIDER': 'vllm',
    'LOCAL_ANALYZER_MODEL_ID': SERVED_MODEL_NAME,
    'INFERENCE_ANALYZER_MODEL': SERVED_MODEL_NAME,
    'LOCAL_ANALYZER_APP_NAME': 'ARC3 Agent Harness',
    
    ########## analyzer config ##########
    'ARC3_REASONING_HISTORY_KEY': 'reasoning_content',
    'LOCAL_ANALYZER_ENABLE_THINKING': 'true',
    'LOCAL_ANALYZER_TOOL_STEPS': '0',
    'LOCAL_ANALYZER_TOOL_TIMEOUT': '30',
    'LOCAL_ANALYZER_YIELD_SECONDS': '0',
    'LOCAL_ANALYZER_YIELD_TOKENS': '2048',
    'LOCAL_ANALYZER_TEMPERATURE': '0.7',
    'LOCAL_ANALYZER_TOP_P': '0.95',
    'LOCAL_ANALYZER_TOP_K': '20',

    ########## context pruning ##########
    # 200K context (2026-10-06): window 188K + 12K reply = 200K, 8K under the server's 208K limit;
    # trim above 192,000 input tokens, drain to 96,000; turn cap scaled with the window.
    'LOCAL_ANALYZER_CONTEXT_WINDOW': (128+12)*1024,  # ctx 140K variant
    'LOCAL_ANALYZER_MAX_OUTPUT': 12*1024,
    'ARC3_CONTEXT_DRAIN_TOKENS': 67000,
    'ARC3_HISTORY_ASSISTANT_TURNS': 170,
    'ARC3_HISTORY_TURN_DRAIN': 34,
    'ARC3_HISTORY_DRAIN_COALESCE': '1',

    ########## tool output ##########
    'LOCAL_ANALYZER_TOOL_OUTPUT_TOKENS': '3072',
    'ARC3_MIDDLE_TRUNCATION': '1',
    'ARC3_ACTION_ECHO': '1',
    
    ########## prompt changes ##########
    'ARC3_EXPLAIN_GAMEPLAY_CHANGED': '1',
    'ARC3_NEW_CHANGED_PROMPTS': '1',
    'ARC3_LEVEL_TRANSFER_GUIDANCE': '1',
    'ARC3_SEARCH_DECOMPOSE': '1',  # subgoal bullet + note on timed-out / out-of-memory calls (since 2026-10-10)
    'ARC3_NEW_ELEMENT_PROBE': '1',  # try every action on unseen elements; goal rarely changes (since 2026-10-10)
    'ARC3_ACTION_INFO': '1',
    'ARC3_DEDUPE_MULTICALL_LINE': '1',
    'ARC3_FRAME_DIFF_HINT': '1',

    ########## vision & animations ##########
    'MULTIMODAL_CONTEXT': 'current_grid',
    'ARC3_PRECACHE_PLE_AHEAD_GB': '40',  # notebook: PLE files read ahead of the loader, at most this many GB
    'ARC3_PRECACHE_RESERVE_GB': '16',  # notebook: memory kept free while reading PLE files ahead
    'ARC3_PRECACHE_PLE_THREADS': '32',  # notebook: threads of the PLE reader (shared by its files)
    'ARC3_PRECACHE_PLE_FILES_IN_FLIGHT': '8',  # notebook: PLE files read at once
    'ARC3_PRECACHE_DROP': '1',  # notebook: drop model files the loader has finished from the page cache
    'MULTIMODAL_UPSCALE': '8',  # 512 x 512 px boards, ~258 vision tokens
    'ARC3_DIFF_IMAGE': '1',
    'ARC3_GAMEOVER_DIFF_IMAGE': '1',
    'ARC3_ANIMATION': '1',

    ########## guards ##########
    'ARC3_GUARDS_FROM_LEVEL': 2,
    'ARC3_BATCH_NOOP_BLOCK': '1',
    'ARC3_SNIPPET_NOOP_LIMIT': '4',
    'ARC3_QSA_FP4_LAYERS': '15-35',  # NVFP4 KV on these full-attention layers  # a snippet stops acting after 4 no-ops in a row
    'ARC3_STALE_STATE_BLOCK': '0',  # off since 2026-10-10: it refused probe loops after a no-op (sp80 red-piece click)

    ########## world model ##########
    'ARC3_MEMORY_SECTIONS': 'off',

    ########## persistent functions ##########
    'ARC3_PERSISTENT_FUNCTIONS': '1',
    'ARC3_PERSISTENT_FUNCTIONS_SCOPE': 'game',
    'ARC3_PERSISTENT_FUNCTIONS_IMPORTS': '1',
    # hard5 experiment: one persistent Python process per game (supersedes retained functions)
    'ARC3_PERSISTENT_PYTHON_PROCESS': '1',
    'ARC3_PYPROC_INVENTORY_ON_TRIM_ONLY': '1',  # list the namespace only at start and after trims
    'ARC3_OPENER_STATIC_ON_TRIM_ONLY': '1',  # opener's fixed instructions only at start and after trims
    'ARC3_PROMPT_NO_SEARCH_URGING': '1',  # drop the 'write a BFS / stop probing and search' wording
    'ARC3_VIEW_SHIFT': '1',  # detect camera scrolls: compensated frame_diff / diff image, echo note
    'ARC3_COMPACT_NOTE': '1',  # goal / action / rule note written at each context trim
    'ARC3_COMPACT_NOTE_STYLE': 'experiments',  # lab notebook: purpose / actions / observed per entry, observations only
    'ARC3_COMPACT_NOTE_MAX_TOKENS': '2000',
    'ARC3_COMPACT_NOTE_IN_SYSTEM': '1',  # the notebook ends the system prompt
    'ARC3_SCHEMA': '0',  # no world model (demo10 no-schema control)
    'ARC3_SCHEMA_STOP': 'call',  # a wrong prediction drops only the rest of its call (v27)
    'ARC3_SCHEMA_TOLERANCE': '16',  # a board difference is wrong from 16 connected cells
    'ARC3_SCHEMA_REPORT_DIFF': '1',  # print frame_diff objects a right prediction missed
    'ARC3_SCHEMA_PARSE': '1',  # WorldModel(parse, step, ...): predictions judged on parsed states
    'ARC3_GRID_MAP': '1',  # grid_map(frame) stub; every message shows grid_map(current_frame)
    'ARC3_GRID_MAP_STEP': '1',  # Workflow step 2: write grid_map(frame) (with cellstat) to build the block map
    'ARC3_SCALE_ECHO': '0',  # off since 2026-10-21: per-action flood fill held the GIL (all25 slowdown)
    'ARC3_PYTHON_PROCESS_TIMEOUT': '30',  # python call limit 30 s (since 2026-10-21; was 90)
    'ARC3_FRAME_DIFF_V2': '1',  # frame_diff without background; frame_diff timelines
    'ARC3_LONG_ANIMATION_NOTE': '1',  # 10+ frame animations name their entry in the echo
    'ARC3_LONG_ANIMATION_FRAMES': '8',  # long-animation note from 8 frames
    'ARC3_MROPE_DEVICE_CACHE': '1',  # SGLang: mrope deltas cached on the GPU (sglang_patches/mrope_cache.py; back on since 2026-10-22)
    'ARC3_STATE_DELTA': '1',  # runtime state in memory, history deltas to the worker
    'ARC3_QSA_INDEXER_BUDGET': '2048',  # SGLang: tokens each QSA query attends (model default 2048; 8192 is the only other)
    'EXPOSE_RESET': 'on',  # RESET offered to the model (system prompt line, RESET-first guard)
    'ARC3_MM_IMAGE_CACHE': '1',  # SGLang: features sent for new images only (sglang_patches/mm_image_cache.py)
    'ARC3_TRACE_GROUPED': '1',  # per-action trace: one line per run of one verdict
    'ARC3_FUNCTION_SOURCE': '1',  # fn_source['name']: source of the model's latest definition
    'ARC3_ACTION_INDEX': '1',  # level_history indices of each batch; per-entry lab note ranges
    'ARC3_LEVEL_TOKEN_NUDGE': '30000',  # stuck-level note from 30K tokens on the level
    'ARC3_LEVEL_TOKEN_NUDGE_EVERY': '30000',  # and every 30K after it
    'ARC3_LEVEL_NUDGE_CAUSES': '1',  # the note lists the usual reasons a level stays unsolved
    'ARC3_GRID_MAP_OPTOUT': '1',  # grid_map reminder: n >= 3 blocks; grid_map = False stops it
    'ARC3_SCALE_NOTE': '1',  # opener notes a scaled object (twice per level); scale echo on
    'ARC3_LOWM_GEMM': '1',  # SGLang: split-K BF16 GEMMs at 33-128 rows (sglang_patches/arc3_lowm_patch.py)
    'ARC3_LOWM_HC_MIX': '1',  # SGLang: HC mix in two launches
    'ARC3_LOWM_HC_COMBINE_NORM': '1',  # SGLang: HC combine fused with the next hc_norm
    'ARC3_ANIM_ONLY_HINT': '1',  # Workflow step 4: animation-only effects keep the state
    'ARC3_DEATH_SUSPECT': '1',  # death notice: objects near the fatal position are first suspects
    'ARC3_SCHEMA_FREE_ACTIONS': '4',  # up to 4 unchecked actions per Python call without a model
    'ARC3_PROMPT_PROBE_ACTIONS': '1',  # system prompt: probe untried SPACE / MOUSE / UNDO early and when stuck
    'ARC3_ACTION_FRAME_IMAGES': '10',  # board after each action of a batch, in order (default since 2026-10-09)
    'ARC3_ANIMATION_REPLAY_FRAMES': '2',  # long animations only: 2 states + final board (since 2026-10-10)
    'ARC3_ANIMATION_REPLAY_MIN': '15',  # ... with >= 15 intermediate states
    'ARC3_DEATH_REPLAY_FRAMES': '5',  # fatal action's animation frames before the death frame (default since 2026-10-09)

    ########## controls ##########
    'EXPOSE_UNDO': 'on',
}

if USE_PRIORITY_SCHEDULING: setup_env.update({
    'ARC3_MAX_ACTIVE_STREAMS': 20,  # 20 concurrent games (MAMBA_CACHE 100 = 20 x 5 slots)
    # KV-budget admission: ARC3_KV_ADMISSION_TOKENS / ARC3_KV_PRESSURE_TOKENS are set in the customization
    # hook from the server's real pool size (90% / 95% of max_total_num_tokens). Pressure valve floor and the
    # premature-trim warning scaled to the 192K trigger; [kv] report every 200 s.
    'ARC3_KV_PRESSURE_MIN_TOKENS': 98000,
    'ARC3_KV_PREMATURE_TOKENS': 119000,
    'ARC3_PRIORITY_REFRESH_QUEUE': '1',
    'ARC3_PRIORITY_PACE': '0',
    'ARC3_PRIORITY_TAIL_FADE': '1',
    'ARC3_PRIORITY_TAIL_FADE_FRACTION': '0.2',
    'ARC3_PRIORITY_TAIL_LOOKUP': 'remaining',
    'ARC3_PRIORITY_SCORE_NORMALIZATION': '1',
})

os.environ.update({k: str(v) for k,v in setup_env.items()})

## 2. Precaching

Precache wheelhouse for faster startup.

## 1b. KLEE 3bpw model and serving config
Serving overrides for the launcher hooks; assembles the klee model directory (MODEL_DIR) before the precache cell reads it.

In [ ]:
# ============================== KLEE 3BPW SERVING CONFIG ==============================
# ISTA variant (serving/ista/build_ista_notebook.py): routed experts from the ISTA GGUF via the ista_moe hook (OFF: W3 baseline)
# Bench variant `klee_w3mid_r25_m125` (github.com/zhangzibo54-cmd/arc3-fast-inference, bench/cell_config.py): the v4 game
# config (NVFP4 main KV + native-FP4 XQA, draft KV fp8, MEMFRAC 0.96, 25 running, 125 Mamba slots
# extra_buffer, no cap2, no skip-decode-lock, chunk 8192, prefill max 3 requests) with the main model replaced by
# klee100/Qwen3.8-Flash-Next-AutoRound-3bpw-MTP served by Humming (bench arc3-klee-3bpw-humming-speed-bench).
V = {'PATCH_NVFP4': True, 'KLEE': True, 'KLEE_HYBRID': False, 'KLEE_PACK_LOCAL': False, 'ISTA': False}  # hybrid A, packed (arc3-kleea-p1..p4): klee100 3bpw with Intel W4 experts in layers 12-35 (W3 ends)
KLEE = False
RESULTS = {}
CFG_OVERRIDES = {'CTX': 148*1024, 'KVDTYPE': 'fp8_e4m3', 'DRAFT_KVDTYPE': 'fp8_e4m3', 'MAXREQ': 20, 'CUDAGRAPH_MAXBS': 20, 'MAMBA_CACHE': 100, 'MAMBA_RADIX': 'extra_buffer', 'MEMFRAC': 0.96, 'CHUNK': 8192, 'MAX_PREFILL': 16384, 'GRAPH_BS': [1, 2, 4, 8, 12, 14, 15, 16, 17, 18, 19, 20]}
ENV_OVERRIDES = {'HUMMING_CACHE_DIR': '/kaggle/working/humming_cache'}
EXTRA_ARGS = ['--prefill-max-requests', '3', '--weight-loader-prefetch-num-threads', '16']
print('CFG_OVERRIDES', CFG_OVERRIDES); print('ENV_OVERRIDES', ENV_OVERRIDES); print('EXTRA_ARGS', EXTRA_ARGS)


In [ ]:
# ============================== KLEE 3BPW MODEL (only used when KLEE) ==============================
# klee100/Qwen3.8-Flash-Next-AutoRound-3bpw-MTP: routed experts W3 g128 (layers 0-11, 36-47) / W2 g64 (12-35),
# QSA/indexer/GDN projections W8 g128, everything else BF16. Mirrored as zhangzibo54/arc3-klee3bpw-p1..p4;
# shard 1 (102 GB BF16 n-gram/PLE table) is not mirrored: assemble.py links the identical Intel PLE-only files.
# Served with `--quantization humming` (SGLang reads per-layer precision only from GPTQModel-style `dynamic` rules,
# built from AutoRound's extra_config by tools/humming_config.py -> config.humming.json in p4).
#  1. assemble a symlinked model dir in /tmp (PLE keys -> Intel files); config.json = Intel architecture config +
#     klee quantization_config (config.humming.json); klee's own layer_types names are unknown to Pennyroyal
#  2. make the `dynamic` rules independent of the module-prefix style: the checkpoint says
#     `model.language_model.layers.N`, SGLang's runtime prefix may be `model.layers.N` (hf_to_sglang mapper).
#     A rule that does not match would silently treat a quantized module as BF16 (or vice versa).
#  3. klee_humming_preflight (called by the launcher inside the venv, before loading 50 GB):
#     resolves every language-model module with SGLang's own HummingConfig, for each prefix style, and compares
#     with the AutoRound precision of the checkpoint -> stops before the load on any mismatch.
import base64 as _base64, hashlib as _hashlib, json as _json, re as _re, sys as _sys, subprocess as _subprocess
from pathlib import Path as _Path

KLEE = bool(V.get('KLEE', False))
KLEE_TOKENIZER_MATCH = True
_LM_LIT = r'model\.language_model\.layers\.'
_LM_ANY = r'(?:model\.)?(?:language_model\.)?(?:model\.)?layers\.'

if KLEE:
    _assemble_path = _Path('/tmp/klee_assemble.py')
    _assemble_path.write_bytes(code_file('assemble.py'))
    _sys.path.insert(0, '/tmp'); _sys.modules.pop('klee_assemble', None)
    from klee_assemble import assemble as _assemble

    INTEL_MODEL_DIR = str(_Path(MODEL_DIR).resolve())
    if V.get('KLEE_PACK_LOCAL', False):
        # hybrid A packed at serve time (bench/klee_model/pack_hybrid.py write_local): only the tensors the
        # hybrid uses are read from the full klee parts + Intel model, written to the local disk, then served
        # as the packed parts (KLEE_HYBRID False)
        _Path('/tmp/klee_pack.py').write_bytes(code_file('pack_hybrid.py'))
        _sys.modules.pop('klee_pack', None)
        from klee_pack import write_local as _write_local
        _pack = _write_local(out='/tmp/kleea-pack/klee3bpw', threads=96)
        RESULTS['klee_pack'] = {k: v for k, v in _pack.items() if k != 'files'}
        MODEL_DIR = _assemble(INTEL_MODEL_DIR, out='/tmp/klee3bpw-model', root='/tmp/kleea-pack')
    else:
        MODEL_DIR = _assemble(INTEL_MODEL_DIR, out='/tmp/klee3bpw-model')
    # hybrid (variant flag KLEE_HYBRID): routed experts of layers 12-35 from the Intel W4 model instead of klee W2;
    # needs the tensor-filter hook (cell_tensorfilter.py) because both checkpoints share tensor names per shard
    KLEE_HYBRID = bool(V.get('KLEE_HYBRID', False))
    _hyb_rule = None
    if KLEE_HYBRID:
        _Path('/tmp/klee_hybrid.py').write_bytes(code_file('hybrid.py'))
        _sys.modules.pop('klee_hybrid', None)
        from klee_hybrid import hybridize as _hybridize
        # HYBRID_SOURCE: 'intel' (Intel W4 g128 experts) or 'w3mid' (our W3 g128 MSE-clip experts, notebooks
        # arc3-w3mid-q1/q2 attached as inputs: directories named w3mid with model.safetensors.index.json)
        _src = V.get('HYBRID_SOURCE', 'intel')
        if _src == 'w3mid':
            _dirs = sorted({str(p.parent) for p in _Path('/kaggle/input').rglob('w3mid/model.safetensors.index.json')})
            assert len(_dirs) == 2, f'expected the two w3mid parts (arc3-w3mid-q1/q2), found {_dirs}'
            _hyb_rule = _hybridize(MODEL_DIR, _dirs, layers=range(12, 36), mid_spec=(3, 128))
        else:
            _hyb_rule = _hybridize(MODEL_DIR, INTEL_MODEL_DIR, layers=range(12, 36))
        print('hybrid source:', _src)
    _mp = _Path(MODEL_DIR)
    _cfg = _json.loads((_mp / 'config.json').read_text())
    _q = _cfg['quantization_config']
    # Architecture fields come from the Intel config (validated on this Pennyroyal build); the klee config uses
    # upstream names that Pennyroyal's Qwen4Exp does not know (layer_types 'qwen_sparse_attention' -> KeyError in
    # qwen3_5.get_layer). Only the quantization_config is taken from klee. Differences are printed for the record.
    _icfg = _json.loads((_Path(INTEL_MODEL_DIR) / 'config.json').read_text())
    def _flat(d, p=''):
        out = {}
        for k, v in d.items():
            if k == 'quantization_config': continue
            out.update(_flat(v, p + k + '.') if isinstance(v, dict) else {p + k: v})
        return out
    _fk, _fi = _flat(_cfg), _flat(_icfg)
    _diff = sorted(k for k in set(_fk) | set(_fi) if _fk.get(k, '<absent>') != _fi.get(k, '<absent>'))
    print(f'klee vs Intel config: {len(_diff)} differing keys (outside quantization_config)')
    for _k in _diff[:60]:
        _a, _b = repr(_fk.get(_k, '<absent>')), repr(_fi.get(_k, '<absent>'))
        print(f'  {_k}: klee={_a[:160]} | intel={_b[:160]}')
    RESULTS['klee_config_diff'] = {k: [str(_fk.get(k, '<absent>'))[:200], str(_fi.get(k, '<absent>'))[:200]] for k in _diff}
    _cfg = dict(_icfg, quantization_config=_q)
    if isinstance(_cfg.get('text_config'), dict) and 'quantization_config' in _cfg['text_config']:
        _cfg['text_config'] = dict(_cfg['text_config'], quantization_config=_q)
    assert _q.get('dynamic') and _q.get('quant_method') == 'auto-round', 'assembled config is not config.humming.json'
    if _hyb_rule is not None:  # first '+' rule wins in Humming; the swapped experts get W4 g128
        _q['dynamic'] = {_hyb_rule[0]: _hyb_rule[1], **{k: v for k, v in _q['dynamic'].items() if k != _hyb_rule[0]}}
    # keep the verified rules (checkpoint prefix) for the preflight, then generalize the prefix
    (_mp / 'HUMMING_RULES_CKPT.json').write_text(_json.dumps(_q))
    _dyn = {k.replace(_LM_LIT, _LM_ANY): v for k, v in _q['dynamic'].items()}
    assert len(_dyn) == len(_q['dynamic'])
    _q['dynamic'] = _dyn
    (_mp / 'config.json').unlink(missing_ok=True)
    (_mp / 'config.json').write_text(_json.dumps(_cfg, indent=1))
    print('quantization_config (top level):', {k: v for k, v in _q.items() if k != 'dynamic'})
    print(f'Humming dynamic rules: {len(_dyn)}'); [print('  ', k, v) for k, v in _dyn.items()]

    # PLE: only symlinks into the Intel dir (loaded to host memory by --ple-offload-embedding)
    _ple = _json.loads((_mp / 'PLE_SOURCE.json').read_text())['files']
    _idx = _json.loads((_mp / 'model.safetensors.index.json').read_text())['weight_map']
    _ple_t = [k for k, f in _idx.items() if f in set(_ple)]
    assert _ple_t and all((_mp / f).is_symlink() and str((_mp / f).resolve()).startswith(INTEL_MODEL_DIR) for f in _ple)
    _shards = sorted(set(_idx.values()))
    _missing = [f for f in _shards if not (_mp / f).is_file()]
    assert not _missing, f'unresolved shards: {_missing}'
    print(f'shards: {len(_shards)} ({len(_ple)} Intel PLE files, {len(_ple_t)} PLE tensors); '
          f'mtp tensors in index: {sum(k.startswith("mtp.") for k in _idx)} (target load skips them; draft = albucino)')
    print('text_config PLE dtype:', _cfg.get('text_config', {}).get('ple_embedding_dtype', 'default'),
          '| architectures:', _cfg.get('architectures'))

    def _sha(p):
        h = _hashlib.sha256()
        with open(p, 'rb') as f:
            for b in iter(lambda: f.read(8 << 20), b''): h.update(b)
        return h.hexdigest()
    for _n in ('tokenizer.json', 'tokenizer_config.json', 'chat_template.jinja', 'generation_config.json'):
        _a, _b = _mp / _n, _Path(INTEL_MODEL_DIR) / _n
        _same = _a.is_file() and _b.is_file() and _sha(_a) == _sha(_b)
        print(f'{_n}: klee {"present" if _a.is_file() else "MISSING"}, same as Intel: {_same}')
        if _n == 'tokenizer.json': KLEE_TOKENIZER_MATCH = _same
    _Path(ENV_OVERRIDES.get('HUMMING_CACHE_DIR', '/kaggle/working/humming_cache')).mkdir(parents=True, exist_ok=True)
    RESULTS['klee'] = dict(model_dir=MODEL_DIR, ple_files=len(_ple), shards=len(_shards),
                           tokenizer_match=KLEE_TOKENIZER_MATCH, dynamic_rules=len(_dyn), hybrid=KLEE_HYBRID)
    if KLEE_HYBRID:
        RESULTS['klee']['hybrid_source'] = _json.loads((_mp / 'HYBRID_SOURCE.json').read_text())

_PREFLIGHT = r'''
import collections, importlib, inspect, json, pkgutil, re, sys, traceback
from pathlib import Path
mp = Path(sys.argv[1])
cfg = json.loads((mp / 'config.json').read_text()); qc = cfg['quantization_config']
ref = json.loads((mp / 'HUMMING_RULES_CKPT.json').read_text())
wmap = json.loads((mp / 'model.safetensors.index.json').read_text())['weight_map']
LM = 'model.language_model.layers.'
FUSE = [(re.compile(r'(.*\.linear_attn)\.in_proj_(qkv|z)$'), r'\1.in_proj_qkvz'),
        (re.compile(r'(.*\.linear_attn)\.in_proj_(b|a)$'), r'\1.in_proj_ba'),
        (re.compile(r'(.*\.self_attn)\.(q|k|v)_proj$'), r'\1.qkv_proj'),
        (re.compile(r'(.*\.mlp\.shared_expert)\.(gate|up)_proj$'), r'\1.gate_up_proj'),
        (re.compile(r'(.*\.mlp\.experts)\.\d+\.(gate|up|down)_proj$'), r'\1')]
def fused(n):
    for p, r in FUSE:
        if p.match(n): return p.sub(r, n)
    return n
def mirror(c, prefix):   # tools/humming_config.py: verified against AutoRound extra_config, 0 mismatches
    if any(e in prefix for e in (c.get('modules_to_not_convert') or [])) or 'lm_head' in prefix: return None
    d = c.get('dynamic', {})
    if any(k[:1] == '-' and re.match(k[2:], prefix) for k in d): return None
    l = next(({**c, **o} for k, o in d.items() if k[:1] == '+' and re.match(k[2:], prefix)), c)
    return (int(l['bits']), int(l['group_size']))
q = {k.rsplit('.', 1)[0] for k in wmap if k.endswith('.qweight')}
w = {k.rsplit('.', 1)[0] for k in wmap if k.endswith('.weight')}
mods = sorted({fused(m) for m in q | w if m.startswith(LM)})
expect = {m: mirror(ref, m) for m in mods}
spec_file = mp / 'EXPECTED_SPECS.json'
if spec_file.exists():  # hybrid: precision derived from the tensor shapes of every shard (klee_hybrid.py)
    shape = {k: (tuple(v) if v else None) for k, v in json.loads(spec_file.read_text()).items()}
    diff = [(m, expect[m], shape.get(m, '<absent>')) for m in mods if shape.get(m, '<absent>') != expect[m]]
    print(f'rules vs tensor shapes: {len(mods)} modules, {len(diff)} mismatches', diff[:6])
    if diff:
        print('PREFLIGHT MISMATCH'); sys.exit(0)
    expect = {m: shape[m] for m in mods}
print('checkpoint modules:', len(mods), collections.Counter(str(v) for v in expect.values()))

import sglang.srt.layers.quantization as Q
HC = None
try:
    HC = Q.get_quantization_config('humming')
except Exception:
    for m in pkgutil.walk_packages(Q.__path__, Q.__name__ + '.'):
        if 'humming' in m.name:
            HC = getattr(importlib.import_module(m.name), 'HummingConfig', None) or HC
print('HummingConfig:', HC, inspect.getsourcefile(HC))
try: print('override_quantization_method ->', HC.override_quantization_method(qc, 'humming'))
except Exception as e: print('override_quantization_method error:', e)
hc = HC.from_config(qc)
meths = [n for n in dir(hc) if not n.startswith('__') and callable(getattr(hc, n))]
print('methods:', meths)
for n in ('is_layer_skipped', 'get_layer_weight_schema', 'get_layer_config', 'get_quant_method'):
    if hasattr(hc, n): print(' ', n, inspect.signature(getattr(hc, n)))
try:  # model class: prefix mapper
    from sglang.srt.models.registry import ModelRegistry
    cls = ModelRegistry.resolve_model_cls(cfg['architectures'])[0]
    mapper = getattr(cls, 'hf_to_sglang_mapper', None)
    print('model class:', cls.__name__, '| hf_to_sglang_mapper:', getattr(mapper, 'orig_to_new_prefix', mapper))
    src = inspect.getsource(sys.modules[cls.__module__])
    print('add_prefix names:', sorted(set(re.findall(r'add_prefix\(\s*["\']([\w\.]+)["\']', src)))[:40])
except Exception as e:
    print('model class lookup failed:', repr(e)[:300])

def bits_of(r):
    for a in ('bits', 'weight_bits', 'num_bits', 'b_bits'):
        v = getattr(r, a, None) if not isinstance(r, dict) else r.get(a)
        if isinstance(v, int): break
    else:
        m = re.search(r'uint(\d)', repr(r)); v = int(m.group(1)) if m else None
    g = getattr(r, 'group_size', None) if not isinstance(r, dict) else r.get('group_size')
    if g is None:
        m = re.search(r'group_size\D{0,3}(\d+)', repr(r)); g = int(m.group(1)) if m else None
    return (v, g)
TWO = len(inspect.signature(hc.is_layer_skipped).parameters) >= 2   # Pennyroyal: (config, prefix)
def resolve(prefix):
    a = (qc, prefix) if TWO else (prefix,)
    if hc.is_layer_skipped(*a): return None
    return bits_of(hc.get_layer_weight_schema(*a))
try:
    for _m in [m for m in mods if m.endswith('.mlp.experts')][:1] + [m for m in mods if m.endswith('.mlp.experts')][-1:]:
        _a = (qc, 'model.layers.' + _m[len(LM):]) if TWO else ('model.layers.' + _m[len(LM):],)
        print('schema repr sample', _m, repr(hc.get_layer_weight_schema(*_a))[:600])
except Exception as e:
    print('schema sample failed:', repr(e)[:300])

status = 'UNVERIFIED'
try:
    styles = ['model.layers.', 'model.language_model.layers.', 'language_model.model.layers.']
    bad = {}
    for s in styles:
        bad[s] = [(m, expect[m], r) for m in mods for r in [resolve(s + m[len(LM):])]
                  if r != expect[m] and not (r is not None and None in r)]   # unparsed schemas are not mismatches
        print(f'style {s:32s}: {len(mods)} modules, {len(bad[s])} mismatches', bad[s][:4])
    parsed = [resolve('model.layers.' + m[len(LM):]) for m in mods]
    if all(r is None or None in r for r in parsed):
        status = 'UNVERIFIED'   # schema attributes not recognized; the rule-vs-shape check above still holds
        print('schema bits/group not parseable; see schema repr sample')
    else:
        status = 'OK' if not any(bad.values()) else 'MISMATCH'
except Exception:
    traceback.print_exc(limit=3)
    s = inspect.getsource(HC).splitlines()
    print('--- HummingConfig source (dynamic / skip / schema lines) ---')
    for i, l in enumerate(s):
        if re.search(r'def |dynamic|skip|schema|prefix|mapper', l): print(f'{i:4d} {l[:150]}')
print('PREFLIGHT', status)
'''

def klee_humming_preflight(python, run_env, model_dir):
    """Run in the Pennyroyal venv before the server loads weights; raise on a precision mismatch."""
    p = _Path('/tmp/klee_preflight.py'); p.write_text(_PREFLIGHT)
    r = _subprocess.run([python, str(p), model_dir], capture_output=True, text=True, env=run_env, timeout=900)
    out = (r.stdout + ('\n' + r.stderr[-3000:] if r.returncode else ''))
    _Path('/kaggle/working/klee_preflight.log').write_text(r.stdout + '\n' + r.stderr)
    print(out[-12000:])
    status = 'ERROR' if r.returncode else (out.rsplit('PREFLIGHT ', 1)[-1].split()[0] if 'PREFLIGHT ' in out else '?')
    RESULTS.setdefault('klee', {})['preflight'] = status
    if status == 'MISMATCH':
        raise RuntimeError('Humming dynamic rules resolve to the wrong precision; see /kaggle/working/klee_preflight.log')
    print('klee preflight:', status, '(UNVERIFIED/ERROR = resolution API not recognized; load continues, '
          'a wrong rule fails at load or in the needle/sanity check)')


In [ ]:
import os
import threading
import time

from concurrent.futures import as_completed, ThreadPoolExecutor

_PRECACHE_DONE = [0]  # bytes read by precache(), for the startup report
def precache(*paths, times=1, delay=0, threads=16) -> None:
    time.sleep(delay)
    print(f'Precaching: Started...')
    start_time = time.time()

    CHUNK_SIZE = 32 * 1024 * 1024  # 128 MB

    chunks = []  # list of (path, offset, length)
    total_size = 0
    file_count = 0

    def _add_file(i: int, path: str) -> None:
        nonlocal total_size, file_count
        size = os.path.getsize(path)
        total_size += size
        file_count += 1
        for offset in range(0, max(size, 1), CHUNK_SIZE):
            length = min(CHUNK_SIZE, size - offset)
            chunks.append((i, path, offset, length))

    for i, path in enumerate(paths):
        try:
            if os.path.isfile(path):
                _add_file(i, path)
                continue
            for root, _, files in os.walk(path):
                for file_name in files:
                    file_path = os.path.join(root, file_name)
                    if os.path.isfile(file_path):
                        _add_file(i, file_path)
        except: pass

    chunks.sort()

    _local = threading.local()

    def _read_chunk(chunk: tuple) -> None:
        try:
            buf = _local.buf
        except AttributeError:
            _local.buf = buf = bytearray(CHUNK_SIZE)
        i, path, offset, _ = chunk
        try:
            with open(path, 'rb') as f:
                f.seek(offset)
                remaining = CHUNK_SIZE
                for i in range(1024):
                    n = f.readinto(buf)
                    if not n:
                        break
                    remaining -= n
                    _PRECACHE_DONE[0] += n
                    if remaining <= 0: break
        except: pass

    with ThreadPoolExecutor(max_workers=threads) as pool:
        for _ in range(times):
            list(pool.map(_read_chunk, chunks))

    elapsed = time.time() - start_time
    print(f'Precaching: loaded {file_count} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')

precache_wheels_thread = threading.Thread(target=precache, args=(WHEELHOUSE_DIR,))
precache_wheels_thread.start()
precache_wheels_thread.join()

# model precache, staged (2026-10-24), started with the server; two readers run side by side:
#  1. the non-PLE model files and the draft model, 32 threads (they go to the GPU);
#  2. the PLE files in name order, a shared pool of ARC3_PRECACHE_PLE_THREADS (32) threads over up to
#     ARC3_PRECACHE_PLE_FILES_IN_FLIGHT (3) files at once (v53: one file at a time with 16 threads read 350 MB/s
#     against 470 MB/s for the non-PLE files at 32). A PLE file starts once (a) the admitted PLE files the loader
#     has not finished stay within ARC3_PRECACHE_PLE_AHEAD_GB (40) and (b) MemAvailable, less our cached files the
#     loader still needs and the non-PLE bytes still to be read, stays above ARC3_PRECACHE_RESERVE_GB (16): the PLE
#     table is allocated as ~100 GB of pinned host memory at model init, so the page cache has ~75 GB in all;
#     the PLE reader starts as soon as the loader's progress frees room, not after the non-PLE reader ends
#     (v53: the PLE files waited until 146 s while dropping began at ~60 s), but not before the loader's first
#     progress line, i.e. after that allocation (2026-10-25: files read at launch were evicted by it);
#  3. meanwhile every model file the server has finished reading is dropped from the page cache
#     (posix_fadvise DONTNEED): non-PLE files sit on the GPU, PLE files in pinned memory. A file counts as
#     finished when serve.log's shard progress is at least its place in the order the server first opened the
#     files plus 2 (prefetch threads open files ahead of the loader) and no server process has it open or
#     mapped. Without /proc access nothing is dropped and step 2 stops at the window.
# ARC3_PRECACHE_DROP='0' keeps the cache; ARC3_PRECACHE_MODEL='0' skips the precache.
import json as _pj
import re as _pre
from pathlib import Path as _PP

_PRE_AHEAD = float(os.environ.get('ARC3_PRECACHE_PLE_AHEAD_GB', '40')) * 1e9
_PRE_RESERVE = float(os.environ.get('ARC3_PRECACHE_RESERVE_GB', '16')) * 1e9
_PRE_DROP = os.environ.get('ARC3_PRECACHE_DROP', '1') != '0'
_PRE_PLE_THREADS = int(os.environ.get('ARC3_PRECACHE_PLE_THREADS', '32'))
_PRE_PLE_FILES = int(os.environ.get('ARC3_PRECACHE_PLE_FILES_IN_FLIGHT', '3'))
_PRE_MARGIN = 2  # shards between a file's open-order place and the progress before it counts as finished
_PRE_LIMIT_S = 1500  # give up the loop after this long (server failed or never reports progress)
_ple_files = []
try:
    _ple_files = sorted(_pj.loads((_PP(MODEL_DIR) / 'PLE_SOURCE.json').read_text())['files'])
except Exception:
    pass
_ple_set = set(_ple_files)
_model_files = sorted(p.name for p in _PP(MODEL_DIR).iterdir() if p.name.endswith('.safetensors'))
_model_size = {n: os.path.getsize(os.path.join(MODEL_DIR, n)) for n in _model_files}
_model_real = {}  # real path -> name (the assembled directory holds symlinks)
for _n in _model_files:
    _model_real[os.path.realpath(os.path.join(MODEL_DIR, _n))] = _n
_precache_paths = [str(p) for p in sorted(_PP(MODEL_DIR).iterdir()) if p.name not in _ple_set] + [DRAFT_MODEL_DIR]


def _pre_tree_size(path):
    if os.path.isfile(path):
        return os.path.getsize(path)
    return sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(path) for f in fs
               if os.path.isfile(os.path.join(r, f)))


_nonple_total = sum(_pre_tree_size(p) for p in _precache_paths)
_PRE = {'order': [], 'gone': set(), 'progress': 0, 'total': 0, 'ple_cached': {}, 'waits': 0.0,
        'dropped_gb': 0.0, 'dropped_nonple': 0, 'ple_read': 0, 'nonple_base': None, 'nonple_done': False,
        'ple_first_s': None, 'tracking': None}


def _pre_server_files():
    """Model files open or mapped by any process but this one (None without /proc access)."""
    used, seen_any = set(), False
    me = os.getpid()
    for pid in os.listdir('/proc'):
        if not pid.isdigit() or int(pid) == me:
            continue
        d = f'/proc/{pid}'
        try:
            for fd in os.listdir(d + '/fd'):
                seen_any = True
                try:
                    target = os.readlink(f'{d}/fd/{fd}')
                except OSError:
                    continue
                if target in _model_real:
                    used.add(_model_real[target])
        except OSError:
            pass
        try:
            with open(d + '/maps') as f:
                for line in f:
                    parts = line.split(None, 5)
                    if len(parts) == 6 and parts[5].strip() in _model_real:
                        used.add(_model_real[parts[5].strip()])
        except OSError:
            pass
    return used if seen_any else None


def _pre_progress():
    """(shards loaded, shards in all) from serve.log's last progress line."""
    try:
        with open(LOG, 'rb') as f:
            f.seek(0, 2)
            f.seek(max(0, f.tell() - 20000))
            text = f.read().decode(errors='replace')
        got = _pre.findall(r'checkpoint shards:\s*\d+% Completed \| (\d+)/(\d+)', text)
        return (int(got[-1][0]), int(got[-1][1])) if got else (0, 0)
    except Exception:
        return 0, 0


def _pre_drop(name):
    path = os.path.join(MODEL_DIR, name)
    try:
        fd = os.open(path, os.O_RDONLY)
        try:
            os.posix_fadvise(fd, 0, 0, os.POSIX_FADV_DONTNEED)
        finally:
            os.close(fd)
        _PRE['dropped_gb'] += _model_size[name] / 1e9
        if name not in _ple_set:
            _PRE['dropped_nonple'] += _model_size[name]
    except Exception:
        pass


def _pre_track(stop):
    """Follow the loader and drop finished files until all shards are loaded (or stop is set)."""
    start = time.time()
    while not stop.is_set() and time.time() - start < _PRE_LIMIT_S:
        used = _pre_server_files()
        _PRE['tracking'] = used is not None
        progress, total = _pre_progress()
        if total == len(_model_files):  # the main model's loader (the draft model has its own count)
            _PRE['progress'], _PRE['total'] = max(_PRE['progress'], progress), total
        if used and len(used) <= 24:  # more at once would be a header scan, not the loader's order
            for name in sorted(used):
                if name not in _PRE['order']:
                    _PRE['order'].append(name)
        if _PRE_DROP and used is not None:
            for place, name in enumerate(_PRE['order']):
                if name in _PRE['gone'] or name in used:
                    continue
                if place + 1 + _PRE_MARGIN <= _PRE['progress'] or _PRE['progress'] >= len(_model_files):
                    _pre_drop(name)
                    _PRE['gone'].add(name)
        if _PRE['total'] and _PRE['progress'] >= _PRE['total'] and len(_PRE['gone']) >= len(_PRE['order']):
            break
        time.sleep(1)


def _pre_room(size):
    """Whether a PLE file of `size` bytes may be read now (window and memory)."""
    ahead = sum(v for n, v in _PRE['ple_cached'].items() if n not in _PRE['gone'])
    nonple_read = max(0, _PRECACHE_DONE[0] - (_PRE['nonple_base'] or 0) - _PRE['ple_read'])
    nonple_cached = max(0, min(nonple_read, _nonple_total) - _PRE['dropped_nonple'])
    nonple_unread = 0 if _PRE['nonple_done'] else max(0, _nonple_total - nonple_read)
    try:
        avail = int(next(l for l in open('/proc/meminfo') if l.startswith('MemAvailable')).split()[1]) * 1024
    except Exception:
        avail = 0
    # MemAvailable counts our own cached files as reclaimable: leave out what the loader still needs
    return (ahead + size <= _PRE_AHEAD
            and avail - ahead - nonple_cached - nonple_unread - size >= _PRE_RESERVE)


def _pre_ple(start):
    """Read the PLE files ahead of the loader (shared pool, several files in flight)."""
    chunk = 32 * 1024 * 1024

    def part(path, offset):
        buf = bytearray(chunk)
        try:
            with open(path, 'rb') as f:
                f.seek(offset)
                n = f.readinto(buf) or 0
            _PRECACHE_DONE[0] += n
            _PRE['ple_read'] += n
        except Exception:
            pass

    def whole(path):
        """One PLE file start to end in 64 MB blocks on one handle (sequential read for the mount)."""
        buf = bytearray(2 * chunk)
        try:
            with open(path, 'rb', buffering=0) as f:
                try:
                    os.posix_fadvise(f.fileno(), 0, 0, os.POSIX_FADV_SEQUENTIAL)
                except Exception:
                    pass
                while True:
                    n = f.readinto(buf)
                    if not n:
                        break
                    _PRECACHE_DONE[0] += n
                    _PRE['ple_read'] += n
        except Exception:
            pass

    waiting = list(_ple_files)
    inflight = {}
    with ThreadPoolExecutor(max_workers=_PRE_PLE_THREADS) as pool:
        while (waiting or inflight) and time.time() - start < _PRE_LIMIT_S:
            for name in [n for n, futs in inflight.items() if all(f.done() for f in futs)]:
                del inflight[name]
            if _PRE['total'] and _PRE['progress'] >= _PRE['total']:
                break  # the load is over
            if waiting and waiting[0] in _PRE['gone']:
                waiting.pop(0)  # the loader is past it
                continue
            if not _PRE['total']:
                # not before the loader's first progress line: the ~100 GB pinned PLE table is allocated at
                # model init and evicts what was cached before it (v63 / kleea-low8 v19-v21: 8 PLE files read
                # at launch were evicted, still counted as cached, filled the window, and the loader read
                # them cold at 10-20 s per shard)
                time.sleep(0.5)
                continue
            if waiting and len(inflight) < _PRE_PLE_FILES and _pre_room(_model_size[waiting[0]]):
                name = waiting.pop(0)
                path = os.path.join(MODEL_DIR, name)
                _PRE['ple_cached'][name] = _model_size[name]
                if _PRE['ple_first_s'] is None:
                    _PRE['ple_first_s'] = time.time() - start
                inflight[name] = [pool.submit(whole, path)]
                continue
            if waiting:
                _PRE['waits'] += 0.5
            time.sleep(0.5)


def _precache_seq(*paths, files=12, block=64 * 1024 * 1024):
    """Read every file once, start to end in large blocks on one handle, `files` files at a time, in order."""
    print(f'Precaching: Started (sequential, {files} files at a time)...')
    start_time = time.time()
    names = []
    for path in paths:
        if os.path.isfile(path):
            names.append(path)
        else:
            for root, _, fs in os.walk(path):
                names += [os.path.join(root, f) for f in sorted(fs) if os.path.isfile(os.path.join(root, f))]
    total = sum(os.path.getsize(p) for p in names)

    def whole(path):
        buf = bytearray(block)
        try:
            with open(path, 'rb', buffering=0) as f:
                try:
                    os.posix_fadvise(f.fileno(), 0, 0, os.POSIX_FADV_SEQUENTIAL)
                except Exception:
                    pass
                while True:
                    n = f.readinto(buf)
                    if not n:
                        break
                    _PRECACHE_DONE[0] += n
        except Exception:
            pass

    with ThreadPoolExecutor(max_workers=files) as pool:
        list(pool.map(whole, names))
    print(f'Precaching: loaded {len(names)} files ({total / 1e9:.2f} GB) in {time.time() - start_time:.2f} seconds.\n')


def _pre_staged():
    stop = threading.Event()
    start = time.time()
    _PRE['nonple_base'] = _PRECACHE_DONE[0]
    tracker = threading.Thread(target=_pre_track, args=(stop,), daemon=True)
    tracker.start()
    ple = threading.Thread(target=_pre_ple, args=(start,), daemon=True)
    ple.start()
    _precache_seq(*_precache_paths, files=24)
    _PRE['nonple_done'] = True
    ple.join(_PRE_LIMIT_S)
    first = _PRE['ple_first_s']
    print(f"Precaching: PLE {len(_PRE['ple_cached'])}/{len(_ple_files)} files read ahead "
          f"({_PRE['ple_read'] / 1e9:.1f} GB, first at {first if first is None else round(first)} s, done at "
          f"{time.time() - start:.0f} s, waited {_PRE['waits']:.0f} s for room); tracking "
          f"{'on' if _PRE['tracking'] else 'OFF'}, dropped {len(_PRE['gone'])} files "
          f"({_PRE['dropped_gb']:.1f} GB) so far", flush=True)
    tracker.join(_PRE_LIMIT_S)  # it keeps dropping finished files until the load ends
    print(f"Precaching: load followed to {_PRE['progress']}/{_PRE['total']} shards; dropped {len(_PRE['gone'])} "
          f"files ({_PRE['dropped_gb']:.1f} GB) from the page cache", flush=True)


if os.environ.get('ARC3_PRECACHE_MODEL', '1') == '0':
    precache_model_thread = threading.Thread(target=lambda: print('Precaching: model directory skipped (ARC3_PRECACHE_MODEL=0)'))
else:
    precache_model_thread = threading.Thread(target=_pre_staged, daemon=True)

## 3. Install the ARC runtime

Install `arc-agi` from the offline competition wheelhouse (the Kaggle submission environment
has no internet).

In [ ]:
# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)

## 4. Import the bundled source

Put the snapshotted repositories on the path (this process and any child processes).

In [ ]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")

## 5. Start serving

Install and startup sglang to serve the model.

## 5a. NVFP4 KV patch
The ported NVFP4-KV patch (cookys/sglang-qsa-nvfp4-kv, Apache-2.0) with native-FP4 XQA decode, applied by the launcher before the server starts.

In [ ]:
# ============================== NVFP4 KV PATCH (only used when PATCH_NVFP4) ==============================
# Source: github.com/cookys/sglang-qsa-nvfp4-kv (Apache-2.0, see NOTICE), developed on Pennyroyal at fb1216c6c.
# Ported to Pennyroyal d00d88e (the wheel used here): 4/5 target files are byte-identical; the chunked-prefill
# history gather in qwen_sparse_attn_backend.py was refactored upstream (single gather_index), so its two anchors
# were re-written for that code. Local fixes (v2): history gather over the joined index (one gather + one
# dequant), and decode/verify dequantize only the batch*stride rows actually read (not the CUDA-graph capacity). Everything else is the original applier, which refuses a tree it does not match.
import base64, json, glob, subprocess
PATCH_NVFP4 = bool(V.get('PATCH_NVFP4', False))
NVFP4_DIR = Path('/tmp/nvfp4patch')
_NVFP4_FILES = {n: code_file(n).decode() for n in ['apply_qsa_nvfp4_patches.py', 'qsa_nvfp4_kv.py', 'test_qsa_nvfp4_gather.py', 'test_concurrent_needle.py', 'NOTICE']}

NVFP4_DIR.mkdir(parents=True, exist_ok=True)
for _name, _text in _NVFP4_FILES.items():
    (NVFP4_DIR / _name).write_text(_text)

def apply_nvfp4_patch(venv, python, run_env):
    srt = sorted(glob.glob(f"{venv}/lib/python*/site-packages/sglang/srt"))[0]
    env_ = dict(os.environ, SGLANG_SRT=srt)
    for args_ in (['--check'], []):
        r = subprocess.run([sys.executable, str(NVFP4_DIR / 'apply_qsa_nvfp4_patches.py'), *args_],
                           env=env_, capture_output=True, text=True)
        print(r.stdout[-3000:], r.stderr[-3000:])
        if r.returncode != 0 and 'already' not in (r.stdout + r.stderr):
            raise RuntimeError('NVFP4 patch failed: ' + ' '.join(args_))
    # standalone GPU unit test (gather/dequant byte-exactness, NaN-scratch guard) before the server takes the GPU
    r = subprocess.run([python, str(NVFP4_DIR / 'test_qsa_nvfp4_gather.py')], capture_output=True, text=True,
                       env=run_env, timeout=1200)
    print('gather test rc', r.returncode); print(r.stdout[-3000:]); print(r.stderr[-3000:])
    RESULTS['nvfp4_gather_test_rc'] = r.returncode
    if r.returncode != 0:
        raise RuntimeError('NVFP4 gather unit test failed')

def run_needle_test(n):
    r = subprocess.run([sys.executable, str(NVFP4_DIR / 'test_concurrent_needle.py'), str(SERVED_MODEL_PORT), str(n), '9000'],
                       capture_output=True, text=True, timeout=1800)
    print(r.stdout[-5000:], r.stderr[-2000:])
    RESULTS['needle_test'] = r.stdout.strip().splitlines()[-1] if r.stdout.strip() else r.stderr[-300:]
print('PATCH_NVFP4 =', PATCH_NVFP4)

# bench hook: optional int8 Mamba-checkpoint support for Qwen4-Exp PLE side states (patches/int8-ckpt-ple/apply_int8_ckpt_ple.py)
INT8_CKPT_PLE = bool(V.get('INT8_CKPT_PLE', False)); import base64 as _b64, hashlib as _hl; _i8src = code_file('int8_ckpt_ple.py'); assert _hl.md5(_i8src).hexdigest() == 'e54f898aac008c89bc05bfeba68e6679', 'int8 patch corrupted'; open('/tmp/i8p.py', 'wb').write(_i8src); _apply_nv_orig = apply_nvfp4_patch; apply_nvfp4_patch = lambda venv, python, run_env: (_apply_nv_orig(venv, python, run_env), print(subprocess.run([sys.executable, '/tmp/i8p.py'], env=dict(os.environ, SGLANG_SRT=sorted(glob.glob(venv + '/lib/python*/site-packages/sglang/srt'))[0]), capture_output=True, text=True)) if INT8_CKPT_PLE else None); print('INT8_CKPT_PLE =', INT8_CKPT_PLE, 'md5 ok')


In [ ]:
# bench hook: index-driven tensor filter for the hybrid klee/Intel checkpoint (variant flag KLEE_HYBRID).
# Appends patches/tensor-filter/tensor_filter_hook.py to sglang/srt/model_loader/loader.py: with
# SGLANG_INDEX_FILTER_DIR=MODEL_DIR every safetensors iterator yields, per shard of MODEL_DIR, only the tensors the
# index maps to that shard (klee W2 experts of layers 12-35 and the non-expert Intel tensors are dropped) and raises
# if a mapped tensor is missing. klee_hybrid_check_load() (after READY) requires "[tensorfilter] OK" in serve.log.
import base64, glob, hashlib, time
from pathlib import Path
KLEE_HYBRID = bool(V.get('KLEE_HYBRID', False))
_tf_src = code_file('tensor_filter.py')
assert hashlib.md5(_tf_src).hexdigest() == '5e73ec123e1078bc6a7bc0709e72a07b', 'tensor_filter hook corrupted'

def _install_tensor_filter(venv):
    f = Path(sorted(glob.glob(f"{venv}/lib/python*/site-packages/sglang/srt/model_loader/loader.py"))[0])
    if b'tensor_filter hook' not in f.read_bytes():
        f.write_bytes(f.read_bytes() + _tf_src)
    print('tensor_filter installed in', f)

_apply_nv_prev_tf = apply_nvfp4_patch
def apply_nvfp4_patch(venv, python, run_env):
    _apply_nv_prev_tf(venv, python, run_env)
    if KLEE_HYBRID:
        _install_tensor_filter(venv)
        run_env['SGLANG_INDEX_FILTER_DIR'] = MODEL_DIR

def klee_hybrid_check_load(log_path, timeout_s=45 * 60):
    """Wait until the main model is loaded; raise unless the tensor filter ran and found every tensor."""
    if not KLEE_HYBRID:
        return
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        text = Path(log_path).read_text(errors='ignore') if Path(log_path).exists() else ''
        lines = [l for l in text.splitlines() if '[tensorfilter]' in l]
        if any('[tensorfilter] OK' in l for l in lines) or proc.poll() is not None:
            break
        time.sleep(10)
    print('\n'.join(l for l in lines if 'kept' not in l or 'OK' in l))
    shard_lines = [l for l in lines if ' kept ' in l and 'OK' not in l]
    print(f'{len(shard_lines)} shard lines; first/last:', shard_lines[:1], shard_lines[-1:])
    RESULTS.setdefault('klee', {})['tensor_filter'] = [l for l in lines if 'OK' in l or 'installed' in l]
    if not any('[tensorfilter] OK' in l for l in lines) or any('not yielded' in l for l in lines):
        raise RuntimeError('hybrid checkpoint: tensor filter did not run or a tensor is missing; see serve.log')
print('KLEE_HYBRID =', KLEE_HYBRID)


## 5b. ISTA codebook experts
Routed experts from the ISTA GSQ-RCO GGUF served by the iq_moe_tc kernels (serving/ista/).

In [ ]:
# ============================== ISTA CODEBOOK EXPERTS (not carried in this notebook) ==============================
# The full cell (ISTA kernels and loader hook) lives in the ista-r25 notebooks. The hard10 notebooks
# run with ISTA off and leave it out to stay under Kaggle's 1 MB notebook limit.
ISTA = bool(V.get('ISTA', False))
assert not ISTA, 'ISTA needs the full ISTA CODEBOOK EXPERTS cell'


def _install_ista(venv, python, run_env):
    raise RuntimeError('ISTA is not carried in this notebook')


def ista_check_load(log_path):
    """Nothing to check: ISTA is off."""


In [ ]:
# Paste this entire file into the launcher cell.
# Required notebook variables: MODEL_DIR, DRAFT_MODEL_DIR, WHEELHOUSE_DIR,
# WORKING_DIR, SERVED_MODEL_NAME, SERVED_MODEL_PORT.
# MODEL_DIR: Intel AutoRound INT4 target, or the assembled klee Humming model.
# DRAFT_MODEL_DIR: downloaded albucino runtime/mtp-int4-g32, or its dataset root.
import os, sys, glob, time, shutil, subprocess, urllib.request, urllib.error
import copy, hashlib, json, re, shlex, socket, zipfile
from pathlib import Path

PREFIX = "/tmp/sgl-intel"       # Separate from the previous fork's /tmp/sgl venv.
LOG = str(Path(WORKING_DIR) / "serve.log")
CFG = dict(
    CTX=(116+12+8)*1024,
    MEMFRAC=0.96,
    MAXREQ=10,
    CUDAGRAPH_MAXBS=10,
    MAMBA_CACHE=60,
    CHUNK=8192,
    MAX_PREFILL=16384,
    SPEC=True,
    FRSPEC=True,
    AUTOTUNE=True,
    # Preserve the original Kaggle checkpoint prefetch/cache behavior.
    PREFETCH_CHECKPOINTS=True,
    DROP_CACHE_AFTER_LOAD=True,
    LINEAR_BACKEND="flashinfer",
    KVDTYPE="fp8_e4m3",
    SSM_DTYPE="bfloat16",
    MAMBA_RADIX="extra_buffer",
    GDN_MTP_CACHE_MODE="none",
    SERVED_NAME=SERVED_MODEL_NAME,
    SPEC_ACCEPT_SINGLE=1.0,
    SPEC_ACCEPT_ACC=1.0,
    SPEC_STEPS=3,
)
# >>> bench hook: variant overrides (empty for baseline)
CFG.update({k: v for k, v in CFG_OVERRIDES.items() if k != "GRAPH_BS"})
print("CFG =", CFG)
VENV = f"{PREFIX}/venv"
PYTHON = f"{VENV}/bin/python"
SGLANG = f"{VENV}/bin/sglang"
TOKEN_MAP_SHA = "becfa41d394b86c26c632bea8f3c6ea64bbb76d7b238d8673c06afae21269f25"
TOKENIZER_SHA = "06b9509352d2af50381ab2247e083b80d32d5c0aba91c272ca9ff729b6a0e523"
EXPERT_TARGET = r"re:^mtp\.layers\.0\.mlp\.experts\.[0-9]+\.(gate_proj|up_proj|down_proj)$"
DENSE_IGNORE = r"re:^(?!mtp\.layers\.0\.mlp\.experts(?:\.|$)).*"


def run(cmd, *, check=True, env=None):
    cmd = list(map(str, cmd))
    print("$", shlex.join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True, env=env)
    if result.stdout.strip(): print(result.stdout.strip()[-6000:])
    if result.stderr.strip() and result.returncode: print(result.stderr.strip()[-6000:])
    if check and result.returncode:
        raise RuntimeError(f"Command failed ({result.returncode}): {shlex.join(cmd)}")
    return result


def find_unique(root, name, *, directory=False, required=True):
    root = Path(root)
    direct = root if root.name == name else root / name
    valid = lambda p: p.is_dir() if directory else p.is_file()
    if valid(direct): return direct
    hits = sorted(p for p in root.rglob(name) if valid(p))
    if len(hits) > 1:
        raise RuntimeError(f"Multiple {name} matches under {root}; use a more specific directory: {hits}")
    if hits: return hits[0]
    if required: raise FileNotFoundError(f"{name} not found under {root}")
    return None


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8*1024*1024), b""): h.update(block)
    return h.hexdigest()


def indexed_shards(root):
    root = Path(root)
    index = json.loads((root / "model.safetensors.index.json").read_text())
    names = sorted(set(index["weight_map"].values()))
    if not names: raise RuntimeError(f"Empty weight index: {root}")
    for name in names:
        rel = Path(name)
        if rel.is_absolute() or ".." in rel.parts:
            raise RuntimeError(f"Invalid shard path: {name}")
        p = root / rel
        if not p.is_file() or p.stat().st_size < 8:
            raise RuntimeError(f"Missing/empty shard: {p}")
    return index, names


def prepare_draft_view(target, source_root, work_root):
    """Adapt metadata only; link original dataset weights without copying them."""
    target, source_root, work_root = map(Path, (target, source_root, work_root))
    configs = [source_root / "config.json"] if (source_root / "config.json").is_file() else []
    configs += sorted(p for p in source_root.rglob("config.json") if p not in configs)
    candidates = []
    for path in configs:
        cfg = json.loads(path.read_text())
        q = cfg.get("quantization_config", {})
        if q.get("quant_method") == "compressed-tensors" and "mtp_routed_experts" in q.get("config_groups", {}):
            candidates.append((path.parent, cfg))
    if len(candidates) != 1:
        raise RuntimeError(f"Expected one INT4 g32 MTP checkpoint under DRAFT_MODEL_DIR={source_root}; found {len(candidates)}")
    source, cfg = candidates[0]
    index, shards = indexed_shards(source)
    if not any(k.startswith("mtp.layers.0.mlp.experts.") for k in index["weight_map"]):
        raise RuntimeError("Draft index does not contain the expected MTP experts")
    cfg = copy.deepcopy(cfg)
    q = cfg["quantization_config"]
    group = q["config_groups"]["mtp_routed_experts"]
    w = group["weights"]
    if (w["num_bits"], w["group_size"], w["symmetric"]) != (4, 32, True):
        raise RuntimeError("Expected symmetric INT4 group32 draft experts")
    if group["targets"] not in (["RoutedExperts"], [EXPERT_TARGET]):
        raise RuntimeError(f"Unexpected draft target rules: {group['targets']}")
    if q.get("ignore", []) not in ([], [DENSE_IGNORE]):
        raise RuntimeError("Unexpected draft ignore rules")
    group["targets"] = [EXPERT_TARGET]
    q["ignore"] = [DENSE_IGNORE]
    # Content/path-specific directory makes notebook reruns idempotent without
    # overwriting an unrelated view. We never modify the read-only datasets.
    identity = json.dumps({"source": str(source.resolve()), "target": str(target.resolve()), "config": cfg}, sort_keys=True)
    suffix = hashlib.sha256(identity.encode()).hexdigest()[:16]
    view = work_root / f"draft-view-{suffix}"
    view.mkdir(parents=True, exist_ok=True)
    config_path = view / "config.json"
    config_text = json.dumps(cfg, indent=2) + "\n"
    if config_path.exists() or config_path.is_symlink():
        if config_path.is_symlink() or config_path.read_text() != config_text:
            raise RuntimeError(f"Unexpected existing draft-view config: {config_path}")
    else:
        config_path.write_text(config_text)
    links = {name: source / name for name in shards}
    links["model.safetensors.index.json"] = source / "model.safetensors.index.json"
    for name in ("tokenizer.json", "tokenizer_config.json", "chat_template.jinja", "preprocessor_config.json", "generation_config.json"):
        if (target / name).is_file(): links[name] = target / name
    for name, src in links.items():
        dest = view / name
        dest.parent.mkdir(parents=True, exist_ok=True)
        if dest.exists() or dest.is_symlink():
            if not dest.is_symlink() or dest.resolve() != src.resolve():
                raise RuntimeError(f"Unexpected existing draft-view file: {dest}")
        else:
            dest.symlink_to(src.resolve())
    print(f"Draft source: {source}\nSGLang draft view: {view} ({len(shards)} linked weight shards)")
    return view


# ---- validate notebook inputs / identify the offline bundle ----
MODEL_DIR = str(Path(MODEL_DIR).expanduser().resolve())
assert Path(MODEL_DIR).is_dir(), f"model dir not found: {MODEL_DIR}"
assert CFG["CUDAGRAPH_MAXBS"] >= CFG["MAXREQ"], "CUDAGRAPH_MAXBS must cover MAXREQ"
assert CFG["CHUNK"] > 0 and CFG["CHUNK"] % 64 == 0
# Refuse to start a second server on an occupied port; do not kill other jobs.
with socket.socket() as probe:
    if probe.connect_ex(("127.0.0.1", int(SERVED_MODEL_PORT))) == 0:
        raise RuntimeError(f"Port {SERVED_MODEL_PORT} is occupied; stop the existing server before rerunning")
Path(PREFIX).mkdir(parents=True, exist_ok=True)
Path(LOG).parent.mkdir(parents=True, exist_ok=True)
WHEELS = find_unique(WHEELHOUSE_DIR, "wheels", directory=True)
LOCK = find_unique(WHEELHOUSE_DIR, "requirements.lock")
sglang_wheels = sorted(WHEELS.glob("sglang-*.whl"))
assert len(sglang_wheels) == 1, f"Expected one SGLang wheel, found {sglang_wheels}"
wheel = sglang_wheels[0]
install_marker = Path(PREFIX) / "installed-bundle.txt"

# ---- one-time offline install, isolated from the old mratsim environment ----
install_env = dict(os.environ, UV_OFFLINE="1", UV_PYTHON_DOWNLOADS="never", PYTHONNOUSERSITE="1")
install_env.pop("PYTHONPATH", None)
if not Path(SGLANG).exists() or not install_marker.is_file():
    print("=== installing Pennyroyal (offline) ===")
    # Bootstrap uv directly from its wheel; no global pip changes/ensurepip.
    uv_wheels = sorted(WHEELS.glob("uv-*.whl"))
    assert len(uv_wheels) == 1, "Expected one bundled uv wheel"
    uv_binary = Path(PREFIX) / "bin/uv"
    uv_binary.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(uv_wheels[0]) as archive:
        members = [n for n in archive.namelist() if Path(n).name == "uv" and not n.endswith("/")]
        assert len(members) == 1, "Cannot identify the Linux uv executable in its wheel"
        binary = archive.read(members[0])
        assert binary.startswith(b"\x7fELF"), "Bundled uv executable is not a Linux ELF binary"
        uv_binary.write_bytes(binary)
    uv_binary.chmod(0o755)
    uv = [str(uv_binary)]
    if not Path(PYTHON).exists():
        run(uv + ["venv", "--python", sys.executable, VENV], env=install_env)
    install = uv + ["pip", "install", "--python", PYTHON, "--no-index", "--find-links", str(WHEELS)]
    run(install + ["-r", str(LOCK)], env=install_env)
    run(install + ["--reinstall", "--no-deps", str(wheel)], env=install_env)
    install_marker.write_text('install')
else:
    print(f"Matching offline bundle already installed at {VENV}")

# ---- bundled CUDA toolkit and linker symlinks ----
cu13 = sorted(Path(VENV).glob("lib/python*/site-packages/nvidia/cu13"))
if not cu13 or not (cu13[0] / "bin/nvcc").is_file():
    raise RuntimeError("Bundled CUDA 13 nvcc missing; check that the complete Pennyroyal wheelhouse was uploaded")
CUDA_HOME = str(cu13[0])
libdir = Path(CUDA_HOME) / "lib"
lib64 = Path(CUDA_HOME) / "lib64"
if not lib64.exists() and not lib64.is_symlink(): lib64.symlink_to("lib")
for so in sorted(libdir.glob("*.so.*")):
    link = libdir / re.sub(r"\.so\..*$", ".so", so.name)
    if not link.exists() and not link.is_symlink(): link.symlink_to(so.name)
assert (libdir / "libcudart.so").exists(), "libcudart.so missing: JIT linker cannot find CUDA runtime"
if not (libdir / "libcuda.so").exists():
    candidates = []
    for directory in ("/usr/local/nvidia/lib64", "/usr/local/nvidia/lib", "/usr/lib/x86_64-linux-gnu", "/usr/lib64"):
        candidates += sorted(p for p in Path(directory).glob("libcuda.so*") if p.is_file())
    if not candidates:
        code = 'import torch; torch.cuda.init(); print(open("/proc/self/maps").read())'
        maps = run([PYTHON, "-c", code], env=install_env).stdout
        candidates = [Path(p) for p in re.findall(r"(/[^\s]*libcuda\.so[^\s]*)", maps)]
    if not candidates: raise RuntimeError("GPU driver libcuda.so not found")
    link = libdir / "libcuda.so"
    if link.is_symlink(): link.unlink()
    link.symlink_to(candidates[0].resolve())
CXX = next((c for c in ("g++-15", "g++-14", "g++-13", "g++-12", "g++-11", "g++") if shutil.which(c)), None)
assert CXX, "Host C++ compiler not found"
CC = CXX.replace("g++", "gcc")
assert shutil.which(CC), f"Matching C compiler missing: {CC}"

# ---- runtime environment ----
env = dict(os.environ)
env.pop("PYTHONPATH", None)
cache = f"{PREFIX}/cache"
for name in ("huggingface", "torch", "torchinductor", "triton", "flashinfer", "cuda", "sglang/jit"):
    Path(cache, name).mkdir(parents=True, exist_ok=True)
env.update({
    "PYTHONNOUSERSITE": "1", "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
    "HF_DATASETS_OFFLINE": "1", "UV_OFFLINE": "1", "UV_PYTHON_DOWNLOADS": "never",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "CUDA_HOME": CUDA_HOME, "CUDACXX": f"{CUDA_HOME}/bin/nvcc",
    "PATH": f"{CUDA_HOME}/bin:{VENV}/bin:" + env.get("PATH", ""),
    "LD_LIBRARY_PATH": f"{CUDA_HOME}/lib64:{CUDA_HOME}/lib:" + env.get("LD_LIBRARY_PATH", ""),
    "CC": CC, "CXX": CXX, "CUDAHOSTCXX": CXX,
    "TORCH_CUDA_ARCH_LIST": "12.0",
    "HF_HOME": f"{cache}/huggingface", "XDG_CACHE_HOME": cache,
    "TORCH_HOME": f"{cache}/torch", "CUDA_CACHE_PATH": f"{cache}/cuda",
    "TORCHINDUCTOR_CACHE_DIR": f"{cache}/torchinductor", "TRITON_CACHE_DIR": f"{cache}/triton",
    "FLASHINFER_WORKSPACE_BASE": f"{cache}/flashinfer",
    "SGLANG_CACHE_DIR": f"{cache}/sglang", "SGLANG_JIT_CACHE_DIR": f"{cache}/sglang/jit",
    "SGLANG_ALLOW_OVERWRITE_LONGER_CONTEXT_LEN": "1",
    "OMP_NUM_THREADS": "8", "MKL_NUM_THREADS": "8", "TOKENIZERS_PARALLELISM": "false",
    "MAX_JOBS": "8", "CMAKE_BUILD_PARALLEL_LEVEL": "8",
    "FLASHINFER_NINJA_JOBS": "8", "FLASHINFER_NVCC_THREADS": "2", "TORCHINDUCTOR_COMPILE_THREADS": "8",
    "SGLANG_ENABLE_SM120_LOWM_BF16_GEMM": "1", "SGLANG_SM120_ONLINE_MXFP8": "0",
    "SGLANG_SM120_LOWM_FP8_WEIGHT": "0", "SGLANG_SM120_LM_HEAD_FP8": "0",
    "SGLANG_MAMBA_CONV_DTYPE": "bfloat16", "SGLANG_NUMA_BIND_V2": "false",
    "SGLANG_MM_PREPROCESS_DEVICE": "cpu", "NUMPY_MADVISE_HUGEPAGE": "0",
})
env.update(ENV_OVERRIDES)  # bench hook
print("ENV overrides:", ENV_OVERRIDES)
run([str(Path(CUDA_HOME) / "bin/nvcc"), "--version"], env=env)
print("Host compiler:", CXX)
# Report the optional later caching patch. This cell does not modify wheel code.
run([PYTHON, "-c", '''import pathlib, sglang, torch
print("sglang", sglang.__version__, "| torch", torch.__version__)
assert torch.cuda.is_available(), "CUDA unavailable"
print("GPU:", torch.cuda.get_device_name(0), "capability:", torch.cuda.get_device_capability(0))
'''], env=env)

# ---- validate target and create the read-only draft's SGLang view ----
target_cfg = json.loads(Path(MODEL_DIR, "config.json").read_text())
q = target_cfg.get("quantization_config", {})
if KLEE:
    assert q.get("dynamic"), "KLEE MODEL_DIR must contain Humming dynamic rules"
    assert q.get("quant_method") == "auto-round", "unexpected KLEE quantization method"
    if not KLEE_TOKENIZER_MATCH:
        CFG["FRSPEC"] = False
        print("KLEE tokenizer differs from the pinned FR-Spec map; disabling FR-Spec")
    klee_humming_preflight(PYTHON, env, MODEL_DIR)  # bench hook: SGLang's Humming precision per module, before the load
else:
    assert q.get("quant_method") == "auto-round" and q.get("bits") == 4, "MODEL_DIR must be the Intel/albucino AutoRound INT4 checkpoint"
_, target_shards = indexed_shards(MODEL_DIR)
print("Target:", MODEL_DIR, "| shards:", len(target_shards), "| PLE dtype:", target_cfg.get("text_config", {}).get("ple_embedding_dtype", "checkpoint default (typically BF16)"))
if KLEE:
    ple_source = json.loads(Path(MODEL_DIR, "PLE_SOURCE.json").read_text())
    print("PLE offload remains enabled; linked source files:", ple_source["files"])
# ARC3_MROPE_DEVICE_CACHE=1: per-request mrope deltas cached on the GPU, so speculative decode steps
# with image requests do not block on a host-to-device copy (game/sglang_patches/mrope_cache.py).
MROPE_CACHE_OLD = '            mrope_deltas = [\n                (\n                    torch.zeros(1, dtype=torch.int64)\n                    if mm_inputs[i] is None\n                    else mm_inputs[i].mrope_position_delta.squeeze(0)\n                )\n                for i in range(batch_size)\n            ]\n            mrope_delta_tensor = torch.stack(mrope_deltas, dim=0).to(device=device)\n'
MROPE_CACHE_NEW = '            # ARC3 patch: keep each request\'s delta on the GPU (copied once per\n            # request) so this per-step path does no host-to-device copy. The\n            # pageable copy blocked the CPU until the GPU drained the verify,\n            # which stalled the overlap scheduler on every decode step.\n            mrope_deltas = []\n            for mm_input in mm_inputs:\n                if mm_input is None:\n                    mrope_deltas.append(torch.zeros(1, dtype=torch.int64, device=device))\n                    continue\n                cache = mm_input.__dict__.setdefault("_arc3_mrope_delta_by_device", {})\n                cached = cache.get(str(device))\n                if cached is None:\n                    cached = mm_input.mrope_position_delta.squeeze(0).to(device=device)\n                    cache[str(device)] = cached\n                mrope_deltas.append(cached)\n            mrope_delta_tensor = torch.stack(mrope_deltas, dim=0)\n'
_fb = sorted(Path(VENV).glob("lib/python*/site-packages/sglang/srt/model_executor/forward_batch_info.py"))
assert len(_fb) == 1, _fb
_src = _fb[0].read_text(encoding="utf-8")
_want_new = os.environ.get("ARC3_MROPE_DEVICE_CACHE", "0") == "1"
_from, _to = (MROPE_CACHE_OLD, MROPE_CACHE_NEW) if _want_new else (MROPE_CACHE_NEW, MROPE_CACHE_OLD)
if _to not in _src:
    assert _src.count(_from) == 1, "forward_batch_info.py changed: the mrope hunk does not match once"
    _fb[0].write_text(_src.replace(_from, _to, 1), encoding="utf-8")
    for _pyc in _fb[0].parent.glob("__pycache__/forward_batch_info.*.pyc"):
        _pyc.unlink()
print("mrope device cache:", "on" if _want_new else "off")
# ARC3_MM_IMAGE_CACHE=1: SGLang processes and sends features only for new images; history images go by content
# hash and are rebuilt in the scheduler only if re-encoded (game/sglang_patches/mm_image_cache.py).
MM_IMAGE_CACHE_TARGETS = [('lib/python*/site-packages/sglang/srt/multimodal/processors/base_processor.py', '# ===== ARC3 mm lazy image features (game/sglang_patches/mm_image_cache.py) BEGIN =====\n', '# ===== ARC3 mm lazy image features END =====\n', '# ===== ARC3 mm lazy image features (game/sglang_patches/mm_image_cache.py) BEGIN =====\n\ndef _arc3_install_mm_lazy_features(base_cls):\n    """Tokenizer side: process and send only new images; items of seen images carry no feature."""\n    import collections\n    import hashlib\n    import io\n    import os\n    import pickle\n    import threading\n\n    import numpy as np\n    import torch\n    from PIL import Image\n    from transformers import BatchFeature\n\n    store = os.environ.get("ARC3_MM_IMAGE_STORE", "/dev/shm/arc3_mm_images")\n    os.makedirs(store, exist_ok=True)\n    lock = threading.Lock()\n    seen = collections.OrderedDict()  # key -> (hash, grid row, rows, stored)\n    meta = {}  # feature width and dtype of this processor\'s output\n    local = threading.local()\n    stats = collections.Counter()\n\n    def image_key(image, kwargs_key):\n        if isinstance(image, Image.Image):\n            data, desc = image.tobytes(), (image.mode, image.size)\n        elif isinstance(image, torch.Tensor):\n            data, desc = image.detach().cpu().contiguous().numpy().tobytes(), (str(image.dtype), tuple(image.shape))\n        elif isinstance(image, np.ndarray):\n            data, desc = np.ascontiguousarray(image).tobytes(), (str(image.dtype), image.shape)\n        else:\n            return None\n        digest = hashlib.blake2b(digest_size=20)\n        digest.update(repr((desc, kwargs_key)).encode())\n        digest.update(data)\n        return digest.digest()\n\n    def store_image(item_hash, inner, image, kwargs):\n        """Write the image (PNG) and the call\'s kwargs for the scheduler; False if not possible."""\n        if not isinstance(image, Image.Image):\n            return False\n        try:\n            proc_dir = os.path.join(store, "processor")\n            if not os.path.exists(os.path.join(proc_dir, "CLASS")):\n                os.makedirs(proc_dir, exist_ok=True)\n                inner.save_pretrained(proc_dir)\n                with open(os.path.join(proc_dir, "CLASS.tmp"), "w") as f:\n                    f.write(type(inner).__name__)\n                os.replace(os.path.join(proc_dir, "CLASS.tmp"), os.path.join(proc_dir, "CLASS"))\n            png = io.BytesIO()\n            image.save(png, format="PNG")\n            path = os.path.join(store, f"{item_hash:016x}.pkl")\n            tmp = f"{path}.{os.getpid()}.{threading.get_ident()}.tmp"\n            with open(tmp, "wb") as f:\n                pickle.dump((type(inner).__name__, kwargs, png.getvalue()), f)\n            os.replace(tmp, path)\n            return True\n        except Exception as e:  # the image is then always sent with features\n            print(f"[arc3 mm lazy features] store failed: {e!r}", flush=True)\n            return False\n\n    class LazyImageProcessor:\n        """The wrapped image processor: features are computed for new images only."""\n\n        def __init__(self, inner):\n            self.__dict__["_arc3_inner"] = inner\n\n        def __getattr__(self, name):\n            return getattr(self.__dict__["_arc3_inner"], name)\n\n        def __setattr__(self, name, value):\n            setattr(self.__dict__["_arc3_inner"], name, value)\n\n        def __call__(self, images=None, **kwargs):\n            inner = self.__dict__["_arc3_inner"]\n            local.info = None\n            items = list(images) if isinstance(images, (list, tuple)) else None\n            if not items or any(isinstance(x, (list, tuple)) for x in items):\n                stats["bypass"] += 1\n                return inner(images=images, **kwargs)\n            kwargs_key = repr(sorted((k, repr(v)) for k, v in kwargs.items()))\n            keys = [image_key(x, kwargs_key) for x in items]\n            if any(k is None for k in keys):\n                stats["bypass"] += 1\n                return inner(images=images, **kwargs)\n            with lock:\n                known = {k: seen[k] for k in set(keys) if k in seen and seen[k][3]}\n            todo = []  # first index of each image that needs features\n            for i, k in enumerate(keys):\n                if k not in known and all(keys[j] != k for j in todo):\n                    todo.append(i)\n            fresh = {}  # key -> (pixel rows, grid row)\n            if todo:\n                out = inner(images=[items[i] for i in todo], **kwargs)\n                pv, grid = out.get("pixel_values"), out.get("image_grid_thw")\n                if (set(out.keys()) != {"pixel_values", "image_grid_thw"} or not isinstance(pv, torch.Tensor)\n                        or not isinstance(grid, torch.Tensor) or pv.device.type != "cpu" or pv.dim() != 2\n                        or len(grid) != len(todo)):\n                    stats["bypass"] += 1\n                    print("[arc3 mm lazy features] unexpected image processor output; this call unpatched", flush=True)\n                    return inner(images=images, **kwargs)\n                meta["width"], meta["dtype"] = pv.shape[1], pv.dtype\n                for i, rows, g in zip(todo, pv.split(grid.prod(-1).tolist()), grid):\n                    fresh[keys[i]] = (rows, g)\n                    h = int.from_bytes(keys[i][:8], "big")\n                    stored = store_image(h, inner, items[i], kwargs)\n                    with lock:\n                        seen[keys[i]] = (h, g.clone(), rows.shape[0], stored)\n                        while len(seen) > 200_000:\n                            seen.popitem(last=False)\n            with lock:\n                info = [seen[k] for k in keys]\n                for k in set(keys):\n                    seen.move_to_end(k)\n            total = sum(x[2] for x in info)\n            pv = torch.empty(total, meta["width"], dtype=meta["dtype"])  # rows of seen images stay unwritten\n            start = 0\n            for k, (_, _, rows, _) in zip(keys, info):\n                if k in fresh:\n                    pv[start:start + rows].copy_(fresh[k][0])\n                start += rows\n            grid = torch.stack([x[1] for x in info])\n            local.info = [(x[0], k in fresh, x[2]) for k, x in zip(keys, info)]\n            stats["calls"] += 1\n            stats["images"] += len(keys)\n            stats["with_features"] += sum(1 for k in keys if k in fresh)\n            if stats["calls"] % 200 == 0:\n                print(f"[arc3 mm lazy features] calls {stats[\'calls\']}, images {stats[\'images\']}, "\n                      f"sent with features {stats[\'with_features\']}, bypass {stats[\'bypass\']}, "\n                      f"seen {len(seen)}", flush=True)\n            return BatchFeature(data={"pixel_values": pv, "image_grid_thw": grid})\n\n    orig_resolve = base_cls._resolve_processor\n\n    def _resolve_processor(self, processor=None):\n        processor, tokenizer = orig_resolve(self, processor)\n        image_processor = getattr(processor, "image_processor", None)\n        if image_processor is not None and not isinstance(image_processor, LazyImageProcessor):\n            processor.image_processor = LazyImageProcessor(image_processor)\n        return processor, tokenizer\n\n    orig_finalize = base_cls._finalize_mm_items\n\n    def _finalize_mm_items(self, mm_items, *, images):\n        info, local.info = getattr(local, "info", None), None\n        if info is not None:\n            image_items = [item for item in mm_items if item.is_image()]\n            if len(image_items) != len(info):\n                raise RuntimeError(f"[arc3 mm lazy features] {len(image_items)} image items for {len(info)} images")\n            for item, (item_hash, has_features, rows) in zip(image_items, info):\n                if not isinstance(item.feature, torch.Tensor) or item.feature.shape[0] != rows:\n                    raise RuntimeError("[arc3 mm lazy features] image item does not match its image")\n                item.hash = item_hash\n                if not has_features:\n                    item.feature = None\n        return orig_finalize(self, mm_items, images=images)\n\n    base_cls._resolve_processor = _resolve_processor\n    base_cls._finalize_mm_items = _finalize_mm_items\n    print(f"[arc3 mm lazy features] tokenizer side installed, store {store}", flush=True)\n\n\n_arc3_install_mm_lazy_features(BaseMultimodalProcessor)\n# ===== ARC3 mm lazy image features END =====\n'), ('lib/python*/site-packages/sglang/srt/managers/mm_schedule.py', '# ===== ARC3 mm lazy image features, scheduler side (game/sglang_patches/mm_image_cache.py) BEGIN =====\n', '# ===== ARC3 mm lazy image features, scheduler side END =====\n', '# ===== ARC3 mm lazy image features, scheduler side (game/sglang_patches/mm_image_cache.py) BEGIN =====\n\ndef _arc3_install_feature_rebuild(module):\n    """Scheduler side: rebuild the features of an image sent without them right before it is encoded."""\n    import io\n    import os\n    import pickle\n\n    import torch\n    import transformers\n    from PIL import Image\n\n    store = os.environ.get("ARC3_MM_IMAGE_STORE", "/dev/shm/arc3_mm_images")\n    processors = {}\n    stats = {"rebuilt": 0}\n\n    def rebuild(item):\n        path = os.path.join(store, f"{item.hash & ((1 << 64) - 1):016x}.pkl")\n        with open(path, "rb") as f:\n            cls_name, kwargs, png = pickle.load(f)\n        if cls_name not in processors:\n            processors[cls_name] = getattr(transformers, cls_name).from_pretrained(os.path.join(store, "processor"))\n        image = Image.open(io.BytesIO(png))\n        image.load()\n        out = processors[cls_name](images=[image], **kwargs)\n        grid = out["image_grid_thw"].reshape(-1)\n        if not torch.equal(grid.to(item.image_grid_thw.device), item.image_grid_thw.reshape(-1)):\n            raise RuntimeError(f"[arc3 mm lazy features] rebuilt grid {grid.tolist()} != item grid")\n        item.feature = out["pixel_values"]\n        stats["rebuilt"] += 1\n        if stats["rebuilt"] % 100 == 1:\n            print(f"[arc3 mm lazy features] rebuilt {stats[\'rebuilt\']} image features in the scheduler", flush=True)\n\n    orig_move = module._move_items_to_device\n\n    def _move_items_to_device(items, device):\n        for item in items:\n            if item.feature is None and item.precomputed_embeddings is None and item.is_image():\n                rebuild(item)\n        return orig_move(items, device)\n\n    module._move_items_to_device = _move_items_to_device\n    # always take the path that moves (and so rebuilds) features before the encoder call\n    module._can_skip_pre_embed_feature_move = lambda data_embedding_func: False\n\n\nimport sys as _a3_sys  # noqa: E402\n\n_arc3_install_feature_rebuild(_a3_sys.modules[__name__])\n# ===== ARC3 mm lazy image features, scheduler side END =====\n')]
for _pattern, _begin, _end, _block in MM_IMAGE_CACHE_TARGETS:
    _paths = sorted(Path(VENV).glob(_pattern))
    assert len(_paths) == 1, _paths
    _src = _paths[0].read_text(encoding="utf-8")
    if _begin in _src:
        _src = _src[:_src.index(_begin)].rstrip("\n") + "\n" + _src[_src.index(_end) + len(_end):]
    if os.environ.get("ARC3_MM_IMAGE_CACHE", "0") == "1":
        _src = _src.rstrip("\n") + "\n\n\n" + _block
    _paths[0].write_text(_src, encoding="utf-8")
    for _pyc in _paths[0].parent.glob(f"__pycache__/{_paths[0].stem}.*.pyc"):
        _pyc.unlink()
print("mm lazy image features:", "on" if os.environ.get("ARC3_MM_IMAGE_CACHE", "0") == "1" else "off")
# ARC3_LOWM_GEMM / _HC_MIX / _HC_COMBINE_NORM: split-K BF16 GEMMs at 33-128 rows, two-launch HC mix, HC combine
# fused with the next hc_norm (game/sglang_patches/arc3_lowm_patch.py, installed as the package arc3_lowm).
import base64 as _b64lw
_LOWM_FILES = {n: code_file(n).decode() for n in ['lowm_splitk_gemm.py', 'hc_mix_2k.py', 'hc_combine_norm.py', 'patch.py']}
_LOWM_ON = any(os.environ.get(_k, "0") == "1" for _k in ("ARC3_LOWM_GEMM", "ARC3_LOWM_HC_MIX", "ARC3_LOWM_HC_COMBINE_NORM"))
_LOWM_HOOK = "\n# ARC3_LOWM_HOOK\ntry:\n    import arc3_lowm.patch as _arc3_lowm_patch\n    _arc3_lowm_patch.{fn}({cls})\nexcept ImportError:\n    pass\n"
_site = sorted(Path(VENV).glob("lib/python*/site-packages"))
assert len(_site) == 1, _site
shutil.rmtree(_site[0] / "arc3_lowm", ignore_errors=True)
if _LOWM_ON:
    (_site[0] / "arc3_lowm").mkdir()
    (_site[0] / "arc3_lowm" / "__init__.py").write_text("")
    for _name, _text in _LOWM_FILES.items():
        (_site[0] / "arc3_lowm" / _name).write_text(
            _text.replace("\nfrom lowm_splitk_gemm import", "\nfrom .lowm_splitk_gemm import"), encoding="utf-8")
for _rel, _fn, _cls in (("srt/layers/quantization/unquant.py", "install_linear", "UnquantizedLinearMethod"),
                        ("srt/layers/hyperconnection.py", "install_hc", "GatedResidual")):
    _path = _site[0] / "sglang" / _rel
    _src = _path.read_text(encoding="utf-8")
    _cut = _src.find("\n# ARC3_LOWM_HOOK\n")
    if _cut >= 0:
        _src = _src[:_cut] + "\n"
    if _LOWM_ON:
        _src = _src.rstrip("\n") + "\n" + _LOWM_HOOK.format(fn=_fn, cls=_cls)
    _path.write_text(_src, encoding="utf-8")
    for _pyc in _path.parent.glob(f"__pycache__/{_path.stem}.*.pyc"):
        _pyc.unlink()
print("arc3 lowm patch:", "on" if _LOWM_ON else "off",
      {_k: os.environ.get(_k, "0") for _k in ("ARC3_LOWM_GEMM", "ARC3_LOWM_HC_MIX", "ARC3_LOWM_HC_COMBINE_NORM")})
DRAFT_VIEW = None
if CFG["SPEC"]:
    assert Path(DRAFT_MODEL_DIR).is_dir(), f"draft dir not found: {DRAFT_MODEL_DIR}"
    DRAFT_VIEW = prepare_draft_view(MODEL_DIR, DRAFT_MODEL_DIR, PREFIX)

# ---- assemble Pennyroyal / AutoRound launch arguments ----
graph_bs = sorted({1, 2, 4, 7, 8, 9, 10, CFG["MAXREQ"], CFG["CUDAGRAPH_MAXBS"]})
graph_bs = [n for n in graph_bs if n <= CFG["CUDAGRAPH_MAXBS"]]
if "GRAPH_BS" in CFG_OVERRIDES: graph_bs = sorted(set(CFG_OVERRIDES["GRAPH_BS"]) | {CFG["MAXREQ"]})  # bench hook
args = [SGLANG, "serve", "--model-path", MODEL_DIR, "--load-format", "safetensors",
    "--model-loader-extra-config", '{"enable_multithread_load":true,"num_threads":8}',
    "--served-model-name", str(CFG["SERVED_NAME"]), "--host", "127.0.0.1",
    "--port", str(SERVED_MODEL_PORT), "--tensor-parallel-size", "1",
    "--dtype", "bfloat16", "--quantization", "humming" if KLEE else "auto-round",
    "--kv-cache-dtype", CFG["KVDTYPE"], "--mem-fraction-static", str(CFG["MEMFRAC"]),
    "--context-length", str(CFG["CTX"]), "--page-size", "64",
    "--max-running-requests", str(CFG["MAXREQ"]), "--chunked-prefill-size", str(CFG["CHUNK"]),
    "--max-prefill-tokens", str(CFG["MAX_PREFILL"]),
    "--cuda-graph-max-bs-decode", str(CFG["CUDAGRAPH_MAXBS"]),
    "--cuda-graph-bs-decode", *map(str, graph_bs),
    "--mamba-ssm-dtype", CFG["SSM_DTYPE"], "--max-mamba-cache-size", str(CFG["MAMBA_CACHE"]),
    "--mamba-radix-cache-strategy", CFG["MAMBA_RADIX"], "--mamba-track-interval", "64",
    "--mamba-backend", CFG["LINEAR_BACKEND"],
    "--linear-attn-decode-backend", CFG["LINEAR_BACKEND"],
    "--linear-attn-prefill-backend", CFG["LINEAR_BACKEND"],
    "--moe-runner-backend", "auto", "--ple-offload-embedding", "--trust-remote-code",
    "--mm-feature-transport", "cpu", "--image-processor-backend", "pil",
    "--reasoning-parser", "qwen3", "--tool-call-parser", "qwen3_coder",
    "--default-chat-template-kwargs", '{"preserve_thinking":true}',
    "--watchdog-timeout", "1800", "--schedule-policy", "lpm", "--warmups", "structured_output",
    "--enable-cache-report", "--enable-metrics", "--enable-request-time-stats-logging"]
if CFG["PREFETCH_CHECKPOINTS"]: args += ["--weight-loader-prefetch-checkpoints"]
if CFG["DROP_CACHE_AFTER_LOAD"]: args += ["--weight-loader-drop-cache-after-load"]
tmpl = Path(MODEL_DIR, "chat_template.jinja")
if tmpl.is_file():
    # Keep the "Reasoning effort ..." line in the system prompt whether or not thinking is on, so a
    # thinking-off request (rolling summaries) shares the cached prefix of the thinking-on game requests.
    # Thinking-on renders are byte-identical to the stock template (serving/check_chat_template.py).
    _t = tmpl.read_text(encoding="utf-8")
    _cond = "{%- if enable_thinking is undefined or enable_thinking is true %}\n    {%- set resolved_reasoning_effort"
    assert _t.count(_cond) == 1, "chat template changed: reasoning-effort condition not found"
    tmpl = Path("/kaggle/working/chat_template_reasoning_line_always.jinja")
    tmpl.write_text(_t.replace(_cond, "{%- if true %}\n    {%- set resolved_reasoning_effort"), encoding="utf-8")
    args += ["--chat-template", str(tmpl)]
if not CFG["AUTOTUNE"]: args += ["--disable-flashinfer-autotune"]
if CFG["GDN_MTP_CACHE_MODE"]: args += ["--gdn-mtp-cache-mode", CFG["GDN_MTP_CACHE_MODE"]]
if CFG["SPEC"]:
    args += ["--speculative-algorithm", "NEXTN", "--speculative-num-steps", str(CFG["SPEC_STEPS"]),
        "--speculative-eagle-topk", "1", "--speculative-num-draft-tokens", str(CFG["SPEC_STEPS"]+1),
        "--speculative-draft-model-path", str(DRAFT_VIEW),
        "--speculative-draft-model-quantization", "compressed-tensors", "--speculative-moe-runner-backend", "auto",
        "--speculative-draft-kv-cache-dtype", CFG.get("DRAFT_KVDTYPE", CFG["KVDTYPE"]),  # bench hook
        "--speculative-accept-threshold-single", str(CFG["SPEC_ACCEPT_SINGLE"]),
        "--speculative-accept-threshold-acc", str(CFG["SPEC_ACCEPT_ACC"])]
    if CFG["FRSPEC"]:
        tok = find_unique(WHEELHOUSE_DIR, "hot_tokens_64k.pt", required=False)
        if tok is None: tok = find_unique(WHEELHOUSE_DIR, "flash-next-64k.pt")
        assert sha256(tok) == TOKEN_MAP_SHA, "Wrong FR-Spec map; use the map from the pinned Pennyroyal bundle"
        assert sha256(Path(MODEL_DIR, "tokenizer.json")) == TOKENIZER_SHA, "Tokenizer differs from the FR-Spec map; set CFG['FRSPEC']=False"
        args += ["--speculative-token-map", str(tok)]

# ARC3_QSA_INDEXER_BUDGET: tokens each QSA query attends (model config indexer_budget = 2048, in
# blocks of 4). SGLang's QSA top-k kernel takes 512 or 2048 blocks, i.e. 2048 or 8192 tokens; the
# override also reaches the MTP draft model (no speculative_draft_model_override_args is set).
_qsa_budget = int(os.environ.get("ARC3_QSA_INDEXER_BUDGET", "2048"))
assert _qsa_budget in (2048, 8192), f"ARC3_QSA_INDEXER_BUDGET must be 2048 or 8192, got {_qsa_budget}"
if _qsa_budget != 2048:
    args += ["--json-model-override-args", json.dumps({"text_config": {"indexer_budget": _qsa_budget}})]
print("QSA indexer budget:", _qsa_budget)
args += list(EXTRA_ARGS)  # bench hook
if PATCH_NVFP4: apply_nvfp4_patch(VENV, PYTHON, env)
else:  # fp8 KV variant: the hooks the NVFP4 patch chain would install
    if KLEE_HYBRID:
        _install_tensor_filter(VENV)
        env['SGLANG_INDEX_FILTER_DIR'] = MODEL_DIR
    if ISTA: _install_ista(VENV, PYTHON, env)  # bench hook: NVFP4 KV patch + GPU unit test
# ARC3_QSA_FP4_LAYERS: NVFP4 KV for the listed QSA full-attention layers, fp8 for the rest
# (game/sglang_patches/arc3_kvmix_patch.py, installed as the package arc3_kvmix; needs the NVFP4 KV patch).
import base64 as _b64km
_KVMIX_SRC = code_file('arc3_kvmix_patch.py').decode()
_KVMIX_ON = bool(os.environ.get("ARC3_QSA_FP4_LAYERS", "").strip())
_KVCALIB_ON = bool(os.environ.get("ARC3_KV_CALIB_DIR", "").strip())  # fp8 calibration statistics
_site = sorted(Path(VENV).glob("lib/python*/site-packages"))
assert len(_site) == 1, _site
shutil.rmtree(_site[0] / "arc3_kvmix", ignore_errors=True)
if _KVMIX_ON:
    assert PATCH_NVFP4, "ARC3_QSA_FP4_LAYERS needs the NVFP4 KV patch (V['PATCH_NVFP4'])"
if _KVMIX_ON or _KVCALIB_ON:
    (_site[0] / "arc3_kvmix").mkdir()
    (_site[0] / "arc3_kvmix" / "__init__.py").write_text("")
    (_site[0] / "arc3_kvmix" / "patch.py").write_text(_KVMIX_SRC, encoding="utf-8")
_KVMIX_HOOK = ("\n# ARC3_KVMIX_HOOK\ntry:\n    import sys as _arc3_sys\n    import arc3_kvmix.patch as _arc3_kvmix\n"
               "    _arc3_kvmix.{fn}(_arc3_sys.modules[__name__])\nexcept ImportError:\n    pass\n")
for _rel, _fn, _on in (("srt/mem_cache/memory_pool.py", "install_pool", _KVMIX_ON),
                       ("srt/model_executor/pool_configurator.py", "install_cell_size", _KVMIX_ON),
                       ("srt/layers/attention/qsa_nvfp4_kv.py", "install_view", _KVMIX_ON),
                       ("srt/layers/attention/qwen_sparse_attn_backend.py", "install_calib", _KVCALIB_ON)):
    _path = _site[0] / "sglang" / _rel
    if not _path.exists():
        assert not _on, f"{_rel} missing (NVFP4 KV patch not applied?)"
        continue
    _src = _path.read_text(encoding="utf-8")
    _cut = _src.find("\n# ARC3_KVMIX_HOOK\n")
    if _cut >= 0:
        _src = _src[:_cut] + "\n"
    if _on:
        _src = _src.rstrip("\n") + "\n" + _KVMIX_HOOK.format(fn=_fn)
    _path.write_text(_src, encoding="utf-8")
    for _pyc in _path.parent.glob(f"__pycache__/{_path.stem}.*.pyc"):
        _pyc.unlink()
print("arc3 kvmix patch:", "on" if _KVMIX_ON else "off", "ARC3_QSA_FP4_LAYERS =",
      repr(os.environ.get("ARC3_QSA_FP4_LAYERS", "")), "| kvcalib:", "on" if _KVCALIB_ON else "off")
# ---- launch detached and wait for health ----
precache_model_thread.start()
print("\n" + shlex.join(args) + "\n")
log_offset = Path(LOG).stat().st_size if Path(LOG).exists() else 0
with open(LOG, "ab", buffering=0) as logf:
    proc = subprocess.Popen(args, env=env, stdout=logf, stderr=subprocess.STDOUT, start_new_session=True)
print(f"pid {proc.pid} -> {LOG}")

def show_log_tail():
    with open(LOG, "rb") as f:
        f.seek(0, 2)
        f.seek(max(log_offset, f.tell()-20000))
        print(f.read().decode(errors="replace"))

url, t0 = f"http://127.0.0.1:{SERVED_MODEL_PORT}/health", time.time()
last_report = -30
def _startup_status(elapsed):
    mem = {}
    try:
        for line in open('/proc/meminfo'):
            k, v = line.split(':', 1)
            mem[k] = int(v.split()[0]) / 1e6
    except Exception:
        pass
    tail = ''
    try:
        with open(LOG, 'rb') as f:
            f.seek(0, 2); f.seek(max(0, f.tell() - 4000))
            t = f.read().decode(errors='replace').replace('\r', '\n').strip().splitlines()
            tail = t[-1][-110:] if t else ''
    except Exception:
        pass
    pre = globals().get('_PRE') or {}
    cached = pre.get('ple_cached', {}) if pre else {}
    ahead = sum(1 for n in cached if n not in pre.get('gone', ()))
    ple = (f"PLE read {len(cached)} (ahead of loader {ahead}), dropped {pre.get('dropped_gb', 0):.0f} GB, "
           f"loader {pre.get('progress', 0)}/{pre.get('total', 0)} | ") if pre else ''
    return (f"  {elapsed}s | precache {_PRECACHE_DONE[0] / 1e9:.1f} GB | {ple}"
            f"MemAvailable {mem.get('MemAvailable', 0):.1f} GB, Cached {mem.get('Cached', 0):.1f} GB | {tail}")
while time.time() - NOTEBOOK_START_TIME < SERVER_STARTUP_TIMEOUT:
    if proc.poll() is not None:
        show_log_tail()
        break
    try:
        with urllib.request.urlopen(url, timeout=5) as r:
            if r.status == 200:
                print(f"\nREADY after {int(time.time()-t0)}s -> {url.replace('/health','/v1')}")
                break
    except Exception:
        pass
    elapsed = int(time.time()-t0)
    if elapsed-last_report >= 30:
        print(_startup_status(elapsed), flush=True); last_report=elapsed
    time.sleep(3)
else:
    print(f"\nDEADLINE at {int(time.time()-NOTEBOOK_START_TIME)}s from notebook start; "
          f"releasing the benchmark with the server still loading")


In [ ]:
# ============================== KLEE HYBRID LOAD CHECK + SANITY ==============================
# 1. serve.log must show the tensor filter ran for every shard of MODEL_DIR with no missing tensor;
# 2. two greedy prompts with known answers (a wrong per-module precision can load and produce garbage).
import json as _json, urllib.request as _ur, time as _time
klee_hybrid_check_load(LOG)
ista_check_load(LOG)  # ISTA variant: every layer loaded
_url = f'http://127.0.0.1:{SERVED_MODEL_PORT}'
_t0 = _time.time()
while _time.time() - _t0 < 45 * 60:
    try:
        if _ur.urlopen(_url + '/health', timeout=5).status == 200: break
    except Exception: pass
    _time.sleep(5)
_qs = [('What is 17*23? Reply with the number only.', '391'),
       ('What is the capital of France? Reply with one word.', 'Paris')]
_ok = []
for _p, _want in _qs:
    _body = _json.dumps(dict(model=SERVED_MODEL_NAME, messages=[{'role': 'user', 'content': _p}],
                             max_tokens=4096, temperature=0.0)).encode()
    try:
        _m = _json.loads(_ur.urlopen(_ur.Request(_url + '/v1/chat/completions', _body,
                                                 {'Content-Type': 'application/json'}), timeout=600).read())['choices'][0]['message']
        _ans = (_m.get('content') or '').strip()
        _ok.append(_want.lower() in (_ans or (_m.get('reasoning_content') or '')[-300:]).lower())
        print(f'Q: {_p}\n  answer: {_ans[:200]!r}  ok={_ok[-1]}')
    except Exception as _e:
        _ok.append(False); print('sanity request failed:', repr(_e)[:300])
print('KLEE sanity:', f'{sum(_ok)}/{len(_ok)}')
if not any(_ok):
    raise RuntimeError('hybrid model answers no sanity prompt correctly; check serve.log / Humming rules')


## 6. Load the benchmark

Unpickle the deployment target and the benchmark, stamping the real submission state onto the
target and pointing the benchmark's outputs at the Kaggle working directory.

In [ ]:
# Restore the deployment target and record the real submission state on it.
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

## 7. Customization hook

Optional: tweak `bm`, `bm.games`, or `bm.solver` here before the run starts — the safe place
for one-off experiments once the deployed bundle has loaded.

In [ ]:
# Make one-off changes to `bm`, `bm.games`, or `bm.solver` here before the run starts.
bm.solver.max_runtime_s_per_game = 48*60 #7920.0
bm.solver.analyzer_timeout = 900.0
bm.solver.concurrency = 10
bm.solver.max_actions_per_game = None
bm.solver.save_request_logs = True
bm.n_passes = 1
# Offline test set. HARD10 = the 10 lowest-scoring games of w3all-r25-1pass-120min (default harness test,
# 10 games x 60 min, all concurrent); None = all 25 games. Ignored in a real submission.
HARD10 = ['g50t', 'cn04', 'lf52', 'tn36', 'bp35', 'sk48', 'wa30', 'sp80', 'su15', 'ls20']  # ar25 dropped (its low score was noise)
DEMO10 = ['ar25', 'ft09', 'lp85', 'r11l', 're86', 'sb26', 'sc25', 'tr87', 'tu93', 'vc33']  # Franzen's original offline set
OFFLINE_GAMES = DEMO10  # hard10 x 60 min, 140K context  # build_ista_notebook --games;  # KV admission test: all 25 games, at most 20 active
OFFLINE_MINUTES = 10
bm.game_weights = None

# KV-budget admission from the server's actual pool (max_total_num_tokens in serve.log): admit under 90%,
# pressure valve above 95%. Read by the harness when the priority gate is first built (first game start).
import re as _re
_pool = int(_re.search(r'max_total_num_tokens=(\d+)', (WORKING_DIR / 'serve.log').read_text(errors='ignore')).group(1))
os.environ['ARC3_KV_ADMISSION_TOKENS'] = str(int(0.90 * _pool))
os.environ['ARC3_KV_ADMISSION_PEAK_TOKENS'] = '0'  # build_ista_notebook --kv-peak
os.environ['ARC3_KV_PRESSURE_TOKENS'] = str(int(0.95 * _pool))
print(f"KV pool {_pool} tokens -> admission {os.environ['ARC3_KV_ADMISSION_TOKENS']}, pressure {os.environ['ARC3_KV_PRESSURE_TOKENS']}")

demo_excluded_games = [] if TRUE_SUBMISSION else []  # KLEE offline check: 25 games

if TRUE_SUBMISSION:  # overrides for contest run
    bm.solver.save_request_logs = False
    bm.n_passes = 1
    if USE_PRIORITY_SCHEDULING:
        bm.solver.concurrency = 120  # game count should be 110
        bm.solver.max_runtime_s_per_game = 532*60
else:
    if USE_PRIORITY_SCHEDULING:
        bm.solver.concurrency = (len(OFFLINE_GAMES) if OFFLINE_GAMES else 25 - len(demo_excluded_games)) * bm.n_passes
        bm.solver.max_runtime_s_per_game = OFFLINE_MINUTES*60
    if float(getattr(target, 'max_runtime_s', 0.0) or 0.0) != 32400.0:
        raise RuntimeError(
            f'Expected the 32400-second notebook budget, got {target.max_runtime_s!r}.'
        )

# show config
print('benchmark.solver:', bm.solver)

## 8. Run the system monitor

Periodically show free main memory, VRAM and disk space.

In [ ]:
# --- system census, one line every 10 minutes -------------------------------
# Everything reads as used/total. RAM is total - MemAvailable, which runs higher
# than the "used" column in `free -h`: that one excludes reclaimable page cache,
# and MemAvailable is the figure that actually predicts an OOM. Disk likewise
# counts the root reserve as unavailable, matching df.
#
# A daemon thread rather than a hook in the run loop: the point of watching a
# long run is to see what happens when it stops behaving, and a print driven by
# the harness stops with it. This keeps reporting through a deadlock.
import os, subprocess, threading, sys

def _sysline(paths=("/", "/kaggle/working")):
    GB = 1024 ** 3
    out = []
    mem = {}
    try:
        for line in open("/proc/meminfo"):
            k, _, rest = line.partition(":")
            p = rest.split()
            if p:
                mem[k] = int(p[0]) * 1024
    except OSError:
        pass
    if mem.get("MemTotal") and mem.get("MemAvailable") is not None:
        out.append(f"RAM {(mem['MemTotal']-mem['MemAvailable'])/GB:.1f}/{mem['MemTotal']/GB:.1f}")
    if mem.get("SwapTotal"):
        out.append(f"swap {(mem['SwapTotal']-mem['SwapFree'])/GB:.1f}/{mem['SwapTotal']/GB:.1f}")
    else:
        out.append("swap none")
    st = {}
    try:
        for line in open("/proc/self/status"):
            k, _, rest = line.partition(":")
            p = rest.split()
            if k in ("VmRSS", "Threads") and p:
                st[k] = int(p[0])
    except OSError:
        pass
    proc = []
    if "VmRSS" in st:
        proc.append(f"{st['VmRSS']*1024/GB:.1f} GB")
    proc.append(f"{st.get('Threads', threading.active_count())} threads")
    try:
        proc.append(f"{len(os.listdir('/proc/self/fd'))} fds")
    except OSError:
        pass
    out.append("proc " + ", ".join(proc))
    try:
        r = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total",
                            "--format=csv,noheader,nounits"],
                           capture_output=True, text=True, timeout=5, check=False)
        gpus = []
        for i, line in enumerate(r.stdout.splitlines()):
            p = [x.strip() for x in line.split(",")]
            if len(p) == 2:
                gpus.append(f"GPU{i} {float(p[0])/1024:.1f}/{float(p[1])/1024:.1f}")
        out.append(" ".join(gpus) if gpus else "GPU unavailable")
    except Exception:
        out.append("GPU unavailable")          # nvidia-smi hangs on a wedged GPU
    disks = []
    for path in paths:
        try:
            s = os.statvfs(path)
        except OSError:
            continue
        total = s.f_blocks * s.f_frsize
        disks.append(f"{path} {(total - s.f_bavail*s.f_frsize)/GB:.1f}/{total/GB:.1f}")
    if disks:
        out.append("disk " + ", ".join(disks))
    return "[sys] " + " | ".join(out)


def start_monitor(interval_minutes=10.0, paths=("/", "/kaggle/working")):
    stop = threading.Event()
    def loop():
        while not stop.is_set():
            try:
                print(_sysline(paths), flush=True)
            except Exception as exc:
                print(f"[sys] census failed: {exc!r}", flush=True)
            stop.wait(interval_minutes * 60.0)
    threading.Thread(target=loop, name="system-stats", daemon=True).start()
    return stop

if not TRUE_SUBMISSION:
    _stop_monitor = start_monitor(2)
    print(_sysline())

## 9. Run the benchmark

In a real competition rerun (`KAGGLE_IS_COMPETITION_RERUN`), wait for the Kaggle gateway and
play the **live competition Arcade**. Otherwise — an interactive "Save & Run" — play the
competition's **bundled environment files offline**, with no gateway required, so the notebook
runs end-to-end without a submission. Teardown commands run afterward even if the run raises.

In [ ]:
print('Starting benchmark...')

# Build the live competition game list from the gateway's available environments.
def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# Build the offline game list from the competition's bundled environment files.
def _offline_games(env_dir: str):
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    arcade = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError(f"No offline environments found under {env_dir}.")
    game_ids = [game_id for game_id in game_ids if not any(game_id.startswith(g) for g in demo_excluded_games)]
    if OFFLINE_GAMES:
        game_ids = [game_id for game_id in game_ids if game_id[:4] in OFFLINE_GAMES]
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# The gateway can take a while to come up; poll until it answers.
def _wait_for_gateway(base_url: str, timeout_s: float = 600.0) -> None:
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")


# Print the run preamble and persist the launcher's git status for diagnostics.
#print((BUNDLE_DIR / "preamble.txt").read_text())
(WORKING_DIR / "git_status.txt").write_text((BUNDLE_DIR / "git_status.txt").read_text())

# arc_agi reads RECORDINGS_DIR and ARC_API_KEY from env (ArcadeSpec carries neither); operation
# mode, environments dir, and base url are all passed explicitly via the spec, so no env is needed.
os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

if TRUE_SUBMISSION:
    # Real submission: play the live competition Arcade served by the Kaggle gateway.
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    # The gateway boots asynchronously; wait before swapping in its game list.
    _wait_for_gateway(os.environ["ARC_BASE_URL"])
    bm.games = _competition_games()
else:
    # Interactive run: play the bundled competition environments offline (no gateway).
    # The competition's environment files ship alongside the wheelhouse in the competition dataset.
    competition_env_files = str(Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels").parent / "environment_files")
    bm.games = _offline_games(competition_env_files)

# Outside a real submission, stop ~10 min before the wall-clock budget for a graceful exit.
soft_end = None
if not TRUE_SUBMISSION:
    budget = float(getattr(target, "max_runtime_s", 0.0) or 0.0)
    if budget > 0:
        soft_end = datetime.fromtimestamp(NOTEBOOK_START_EPOCH) + timedelta(seconds=budget - min(600.0, budget / 2))

# Play the benchmark; teardown commands run even if the run raises.
await bm.run(soft_end_time=soft_end, runtime_environment=target, minimal_diagnostics=TRUE_SUBMISSION)
if not TRUE_SUBMISSION:
    # An offline run isn't scored, but Kaggle still expects a submission.parquet output.
    import pandas as pd

    pd.DataFrame(
        [["1_0", "1", True, 1]],
        columns=["row_id", "game_id", "end_of_game", "score"],
    ).to_parquet(WORKING_DIR / "submission.parquet", index=False)

## 10. Show the diagnostics

A non-submission run writes `diagnostics.html` to `/kaggle/working`; it is rendered inline below
(and downloadable from the working directory). You should be able to click around through the links.

In [ ]:
from html import escape

from IPython.display import HTML, display

diagnostics_html = WORKING_DIR / "diagnostics.html"
if diagnostics_html.is_file():
    # Isolate the full document in an iframe so its styles don't leak into the notebook.
    display(
        HTML(
            f'<iframe srcdoc="{escape(diagnostics_html.read_text(), quote=True)}" '
            'width="100%" height="900" style="border:0"></iframe>'
        )
    )
else:
    print("No diagnostics.html — minimal diagnostics (real submission) suppresses it.")